# 第 83 课：推理服务变更管理、配置漂移与端到端恢复验收

**从“模型恢复成功”走到“服务的实际配置与发布意图也一致”。**

承接第 81–82 课的 WAL、双副本、真实 CPU KV 与恢复演练，这一课加入：

- 声明式配置和严格 schema：模型/模板/WAL身份、路由权重、租户白名单、会话上限、KV紧急余量
- 发布前证明：配置差异、真实KV探针、双副本容量、ACK/epoch/旧版本绑定、逻辑TTL
- 原子活动指针：双副本先预置，串行 CAS 后才生效；单个预置失败不能造成半发布
- 漂移检测与显式修复：任何副本观测配置不一致均暂停新请求与自动接管
- 端到端故障恢复：WAL中的请求版本保持固定，物理KV从token重算，旧epoch被围栏

**默认离线、CPU、PyTorch。** 本课真实运行随机微型因果 Transformer 的 Q/V LoRA 与物理KV张量，不下载模型、不访问云平台、无需上一课源码。

## 0. 学习准备与“演练”和“生产”的区别

前置：第 70 课的发布与回滚、第 73 课的 KV、第 81 课的 WAL 恢复、第 82 课的故障预算。

使用当前已有 PyTorch 环境，从上到下执行所有单元。脚本与 Notebook 均自包含，均可单独验证。默认完整演练开启、练习测试关闭；Notebook 在自己的工作目录下创建 `lesson_83_outputs/run_时间戳/`，不会覆盖之前的数据。

我们只用**单进程串行控制器＋两个独立本地目录**来说明代际和持久化指针。磁盘 `fsync` 不是远程同步ACK；Hash链＋本地head没有外部可信锚点；用 `{'approved': True}` 的普通字典不代表真实IAM身份认证；逻辑tick不等于生产RTO秒数。不能直接迁移为生产自动发版控制平面。

对真实 Kubernetes 集群，Server-Side Apply 的字段管理和冲突检测由 API Server 承担，本课没有复现它的多写入方一致性机制。[Kubernetes Server-Side Apply](https://kubernetes.io/docs/reference/using-api/server-side-apply/) · [NIST 配置管理](https://csrc.nist.gov/pubs/sp/800/128/upd1/final)

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第83课：推理服务变更管理、配置漂移与端到端恢复验收。

完整离线、CPU PyTorch：真实自回归Transformer、Q/V LoRA与物理KV张量。
承接双副本WAL与物理KV，演示声明式配置、CAS发布、漂移阻断与恢复后验收。
仅为单进程模拟的双实例；不提供真实集群共识、远端存储、网络或IAM。
所有RTO/RPO单位是显式的教学逻辑tick；不宣称GPU毫秒性能。
"""
from __future__ import annotations

# %% 01 环境与适用边界
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要已有PyTorch环境；不会自动安装或下载。') from exc

VERSION = 'lesson83-config-drift-gates-v1'
SOURCES = {
    'k8s_server_side_apply': 'https://kubernetes.io/docs/reference/using-api/server-side-apply/',
    'nist_config_management': 'https://csrc.nist.gov/pubs/sp/800/128/upd1/final',
    'sre_canary': 'https://sre.google/workbook/canarying-releases/',
    'sre_change': 'https://sre.google/sre-book/reliable-product-launches/',
    'torch_random': 'https://docs.pytorch.org/docs/stable/notes/randomness.html',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
RUN_FULL_LAB = True
OUTPUT_ROOT = Path('lesson_83_outputs')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('本课默认设备：CPU；没有对RTX 3060或真实云集群做性能实测。')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA: False
本课默认设备：CPU；没有对RTX 3060或真实云集群做性能实测。


## 1. 分清三种身份：artifact、意图配置、已生效配置

服务运行时至少有三个事实：

1. **Artifact 身份**：真正的 tokenizer/chat template、权重、模型结构。
2. **期望意图**：这次发布希望启用的版本、路由百分比、租户和资源上限。
3. **运行观测**：两个副本实际加载/应用的配置，以及持久活动指针。

不能只检查期望 JSON 是否正确，更不能只对齐模型文件名。实际身份使用配置与模型权重的内容摘要，发布必须验证双副本观测值一致。

本课自建 UTF-8 字节/小合并的教学 tokenizer 和 Chat Template：控制 token ID 独立于用户文本，不能冒充真实 Qwen 或任意 Hugging Face Tokenizer。

In [3]:
# %% 02 教学分词器、因果模型与LoRA（自包含）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

## 2. 为什么必须验证物理 KV，而不只是逻辑会话列表？

本课使用真正的 PyTorch Tensor 作为分页 KV 存储：每次增量 decode 将该 token 的 K/V 写入缓存，按块表读取历史，计算当前 token 的注意力输出。

核心不变量：**同一模型版本、同一完整 token 前缀，增量 KV 的下一 token logits 必须和全上下文前向一致**。终止或删除会话时，引用计数归零、物理块清零。

单请求最大上下文等约束由模型与池检查，OOM 不会被伪装为 EOS 或成功恢复。这个教学 KV 池不是 vLLM 的生产分页内核，也不是 GPU 显存评测。

In [4]:
# %% 03 实际KV物理缓存与增量重算（不直接持久化tensor）
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u


class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

### 实际模型验证的边界

即使两个 LoRA 版本共享同一基座，也不能跨版本复用同一批 KV：Q/V 权重不同，就可能改变历史注意力表征。此处模型是**随机初始化的演示模型**，只验证数值执行路径，不报告生成质量。

In [5]:
# %% 04 双版本确定性模型与不可变身份
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

In [6]:
seed_all(8301)
tok, versions = build_servable_models(8301)
print('模型版本:', list(versions))
print('权重不同:', state_hash(versions['stable'].state_dict()) != state_hash(versions['canary'].state_dict()))
print('模型配置:', asdict(versions['stable'].cfg))
print('用户消息的教学模板:', tok.apply_chat_template([{'role':'user','content':'hello'}], add_generation_prompt=True))

模型版本: ['stable', 'canary']
权重不同: True
模型配置: {'vocab_size': 281, 'dim': 16, 'heads': 2, 'layers': 1, 'ff_dim': 32, 'max_length': 96}
用户消息的教学模板: [1, 3, 6, 275, 4, 3, 7]


## 3. WAL 固定请求版本，配置升级不能改动已经开始的回答

请求创建记录在持久 WAL 中包含：请求 ID、租户、幂等键、选中的模型版本、prompt IDs、生成长度预算。新请求按当前活动版本配置路由，但重试同一幂等键必须从 WAL 中读取原模型版本。

例如：

```text
第 1 代：灰度流量 0% → 第一次请求使用 stable
第 2 代：灰度流量100% → 同键重试仍用 stable
第 2 代：新幂等键 → 新请求用 canary
```

更重要的是，新版本配置必须与 token 会话绑定，而不是在每一个 decode token 时重新随机路由。

WAL的单机 `fsync`、双副本ACK 和控制代际处理均在独立代码里；这不是网络容错/分布式一致性协议。

In [7]:
# %% 05 WAL逻辑事件、完整性检查与快照

ZERO_HASH='0'*64

@dataclass
class LogicalRequest:
    rid: str
    tenant: str
    key: str
    version: str
    prompt_ids: list[int]
    max_new_tokens: int
    generated: list[int]=field(default_factory=list)
    terminal: str|None=None

    def identity(self):
        return {'tenant':self.tenant,'key':self.key,'version':self.version,
                'prompt_ids':self.prompt_ids,'max_new_tokens':self.max_new_tokens}


class Ledger:
    """WAL顺序应用的逻辑状态，不包含任何模型权重或物理KV。"""
    def __init__(self,codec:ByteMergeChatTokenizer,model_identity:dict):
        self.codec=codec
        self.model_identity=copy.deepcopy(model_identity)
        self.sessions:dict[str,LogicalRequest]={}
        self.seq=0;self.last_hash=ZERO_HASH;self.tick=0

    def to_json(self):
        return {'seq':self.seq,'last_hash':self.last_hash,'tick':self.tick,
                'sessions':{key:asdict(state) for key,state in sorted(self.sessions.items())}}

    def apply(self,record:dict):
        require(type(record) is dict and set(record)=={'seq','tick','kind','payload','prev','hash'},
                'WAL字段集合错误')
        require(type(record['seq']) is int and record['seq']==self.seq+1,'WAL序号必须连续')
        require(type(record['tick']) is int and record['tick']==self.tick+1,
                '本课tick是从1开始的每条持久事件序号')
        require(record['prev']==self.last_hash,'WAL前驱hash错误或从错误分叉接入')
        require(record['hash']==digest({k:v for k,v in record.items() if k!='hash'}),
                'WAL内容hash错误')
        kind=record['kind'];p=record['payload']
        require(type(p) is dict,'payload必须是dict')
        if kind=='create':
            require(set(p)=={'rid','tenant','key','version','prompt_ids','max_new_tokens'},
                    'create字段集合错误')
            rid=make_rid(p['tenant'],p['key'])
            require(rid==p['rid'] and rid not in self.sessions,'重复idempotency key/create冲突')
            require(p['version'] in self.model_identity['weights'],'WAL引用未知权重版本')
            require(type(p['prompt_ids']) is list and len(p['prompt_ids'])>0 and
                    len(p['prompt_ids'])<self.model_identity['max_length'], '提示token非法/超上下文')
            require(all(type(t) is int and 0<=t<self.codec.vocab_size for t in p['prompt_ids']),
                    'WAL提示token超词表')
            require(type(p['max_new_tokens']) is int and 1<=p['max_new_tokens']<=4 and
                    len(p['prompt_ids'])+p['max_new_tokens']<=self.model_identity['max_length'],
                    'WAL完成长度非法')
            self.sessions[rid]=LogicalRequest(**p)
        elif kind=='token':
            require(set(p)=={'rid','index','token','terminal'},'token字段集合错误')
            rid=p['rid'];require(rid in self.sessions,'追加token的请求不存在')
            session=self.sessions[rid]
            require(session.terminal is None,'终态后不可再追加token')
            require(type(p['index']) is int and p['index']==len(session.generated),
                    'token序号不连续')
            tok=p['token'];require(type(tok) is int and 0<=tok<self.codec.vocab_size,
                                  'token ID越界')
            require(p['terminal'] in {None,'eos','length'},'非法停止原因')
            next_length=len(session.generated)+1
            expected='eos' if tok==self.codec.eos_token_id else (
                     'length' if next_length==session.max_new_tokens else None)
            require(p['terminal']==expected,'EOS、长度终止不一致')
            session.generated.append(tok)
            session.terminal=p['terminal']
        else:
            raise LessonError('WAL事件类型非法')
        self.seq=record['seq'];self.tick=record['tick'];self.last_hash=record['hash']

    @classmethod
    def from_json(cls,codec,model_identity,data):
        require(type(data) is dict and set(data)=={'seq','last_hash','tick','sessions'},
                '快照状态字段错误')
        out=cls(codec,model_identity)
        require(type(data['seq']) is int and data['seq']>=0 and
                data['tick']==data['seq'] and type(data['last_hash']) is str and
                len(data['last_hash'])==64,'快照head错误')
        out.seq=data['seq'];out.tick=data['tick'];out.last_hash=data['last_hash']
        require(type(data['sessions']) is dict,'快照会话表必须为dict')
        for rid,s in data['sessions'].items():
            require(type(s) is dict and set(s)==set(LogicalRequest.__dataclass_fields__),
                    '快照会话字段非法')
            item=LogicalRequest(**s)
            require(item.rid==rid==make_rid(item.tenant,item.key),'快照id不一致')
            require(item.version in model_identity['weights'],'快照版本已失效')
            require(item.terminal in {None,'eos','length'},'快照停止状态非法')
            require(all(type(i) is int and 0<=i<codec.vocab_size for i in
                        item.prompt_ids+item.generated),'快照token越界')
            require(1<=item.max_new_tokens<=4 and
                    len(item.generated)<=item.max_new_tokens,'快照生成长度非法')
            if item.terminal=='eos':
                require(bool(item.generated) and item.generated[-1]==codec.eos_token_id,
                        '快照EOS状态不一致')
            if item.terminal=='length':
                require(len(item.generated)==item.max_new_tokens and
                        item.generated[-1]!=codec.eos_token_id,'快照长度终止不一致')
            if item.terminal is None:
                require(codec.eos_token_id not in item.generated and
                        len(item.generated)<item.max_new_tokens,'活动快照包含终态回答')
            out.sessions[rid]=item
        return out


def make_rid(tenant:str,key:str) -> str:
    for value,label in ((tenant,'租户'),(key,'幂等键')):
        require(type(value) is str and re.fullmatch('[A-Za-z0-9_-]{1,18}',value) is not None,
                label+'只能含1..18个ASCII字母数字_-')
    return tenant+'__'+key


def atomic_json(path:Path,payload:dict) -> None:
    """单机持久文件替换＋fsync；没有提供跨机器线性一致性。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    require(not tmp.exists(),'存在未清理的临时文件，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(payload,ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def make_record(ledger:Ledger,kind:str,payload:dict) -> dict:
    row={'seq':ledger.seq+1,'tick':ledger.tick+1,'kind':kind,
         'payload':copy.deepcopy(payload),'prev':ledger.last_hash}
    row['hash']=digest(row)
    trial=copy.deepcopy(ledger)
    trial.apply(row)
    return row


class DurableJournal:
    """严格顺序WAL：每行必须换行、hash chain，追加时flush+fsync。

    这是本地模拟“两个独立实例的磁盘”，不是分布式事务，不跨网复制fsync保证。
    故障后的未完成行默认拒绝；不静默截断或忽略。
    """
    def __init__(self,folder:Path,codec,model_identity,load_existing=False):
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.path=self.folder/'events.jsonl'
        self.snapshot_path=self.folder/'snapshot.json'
        self.codec=codec;self.model_identity=copy.deepcopy(model_identity)
        if not load_existing:
            require(not self.path.exists() and not self.snapshot_path.exists(),
                    '实例目录已有数据，禁止当作新副本覆盖')
            self.path.write_bytes(b'')
        else:
            require(self.path.is_file(),'恢复时WAL不存在')
        self.records=[];self.ledger=Ledger(codec,model_identity)
        if load_existing:self.recover()

    def read_records(self) -> list[dict]:
        raw=self.path.read_bytes()
        require(not raw or raw.endswith(b'\n'),'WAL末尾存在截断行，必须人工修复/显式恢复流程')
        rows=[];temp=Ledger(self.codec,self.model_identity)
        for n,line in enumerate(raw.splitlines(),1):
            require(line!=b'','WAL存在空行')
            try:
                row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError(f'WAL第{n}行不是有效JSON/UTF8') from exc
            temp.apply(row)
            rows.append(row)
        return rows

    def recover(self) -> dict:
        rows=self.read_records()  # 对全WAL做语法/hash链验证，快照不绕过完整性审计。
        all_state=Ledger(self.codec,self.model_identity)
        for row in rows:all_state.apply(row)
        snapshot_used=False;tail=rows
        if self.snapshot_path.exists():
            try:wrapped=json.loads(self.snapshot_path.read_text(encoding='utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('快照文件无法解码') from exc
            require(type(wrapped) is dict and set(wrapped)=={'payload','sha256'},'快照封装非法')
            data=wrapped['payload']
            require(wrapped['sha256']==digest(data),'快照内容摘要不匹配')
            require(type(data) is dict and set(data)=={'version','model_identity','state'},
                    '快照字段错误')
            require(data['version']==VERSION and data['model_identity']==self.model_identity,
                    '快照模型/分词身份不匹配')
            restored=Ledger.from_json(self.codec,self.model_identity,data['state'])
            require(restored.seq<=len(rows),'快照seq超出WAL实际长度')
            prefix=Ledger(self.codec,self.model_identity)
            for row in rows[:restored.seq]:prefix.apply(row)
            require(prefix.to_json()==restored.to_json(),
                    '快照与WAL前缀的内容不一致：不允许信任重算hash后的假快照')
            tail=rows[restored.seq:]
            for row in tail:restored.apply(row)
            require(restored.to_json()==all_state.to_json(),'快照加尾WAL不等于全量恢复')
            self.ledger=restored;snapshot_used=True
        else:self.ledger=all_state
        self.records=rows
        return {'wal_records_verified':len(rows),'snapshot_used':snapshot_used,
                'tail_events_semantically_applied':len(tail),'head_hash':self.ledger.last_hash}

    def append_row(self,row:dict) -> None:
        # 在触盘前先检查下一状态，避免错误事件落盘后再被拒绝。
        probe=copy.deepcopy(self.ledger);probe.apply(row)
        with self.path.open('ab') as f:
            payload=(canonical(row)+'\n').encode('utf-8')
            f.write(payload);f.flush();os.fsync(f.fileno())
        self.ledger=probe
        self.records.append(copy.deepcopy(row))

    def checkpoint(self) -> dict:
        data={'version':VERSION,'model_identity':copy.deepcopy(self.model_identity),
              'state':self.ledger.to_json()}
        atomic_json(self.snapshot_path,{'payload':data,'sha256':digest(data)})
        return {'seq':self.ledger.seq,'last_hash':self.ledger.last_hash,
                'size_bytes':self.snapshot_path.stat().st_size,
                'contents':'logical token IDs and request states; NOT raw KV tensors'}

In [8]:
# %% 06 独立推理副本：WAL恢复后重新prefill真实KV

def artifact_identity(tok,models:dict[str,TinyChatLM]) -> dict:
    require(type(models) is dict and set(models)=={'stable','canary'},'模型版本集合不完整')
    configs=[asdict(m.cfg) for m in models.values()]
    require(configs[0]==configs[1],'两个版本配置不同')
    return {'tokenizer':digest(tok.contract()),'weights':{k:state_hash(m.state_dict())
              for k,m in sorted(models.items())},'config':configs[0],
              'max_length':configs[0]['max_length']}


class Replica:
    def __init__(self,name:str,folder:Path,tok,models,capacity_blocks=64,
                 block_size=4,load_existing=False):
        require(name in {'A','B'},'副本名必须A/B')
        self.name=name;self.tok=tok;self.models=copy.deepcopy(models)
        self.identity=artifact_identity(tok,self.models)
        self.folder=Path(folder)
        self.journal=DurableJournal(self.folder,tok,self.identity,load_existing)
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(capacity_blocks,block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(tok,self.models,self.pool)
        self.up=True
        self.last_recovery={'wal_records_verified':0,'snapshot_used':False,
                            'tail_events_semantically_applied':0}
        if load_existing:self.recover_cache()

    @property
    def ledger(self):return self.journal.ledger

    def _expected(self,rid):
        require(rid in self.pool.sessions and rid in self.runner.last_logits,
                '活动请求没有对应真实KV；拒绝偷偷从空上下文解码')
        logits=self.runner.last_logits[rid]
        require(bool(torch.isfinite(logits).all()),'KV输出异常')
        return int(logits.argmax())

    def prevalidate(self,row:dict) -> None:
        require(self.up,'实例已宕机')
        require(artifact_identity(self.tok,self.models)==self.identity,
                '运行模型权重漂移，拒绝生成/应用新事件')
        test=copy.deepcopy(self.ledger);test.apply(row)
        kind=row['kind'];payload=row['payload']
        if kind=='token':
            rid=payload['rid']
            require(rid in self.pool.sessions and rid in self.runner.last_logits,
                    '副本缺失活动KV：先恢复再继续')
            require(self._expected(rid)==payload['token'],
                    '复制到另一个副本时增量输出不一致：权重/前缀/位置存在漂移')

    def accept(self,row:dict) -> None:
        self.prevalidate(row)
        self.journal.append_row(row)
        p=row['payload']
        try:
            if row['kind']=='create':
                self.runner.open_prefill(p['rid'],p['version'],p['prompt_ids'])
            else:
                if p['terminal'] is None:self.runner.step(p['rid'],p['token'])
                else:self.runner.close(p['rid'])
            self.pool.audit()
        except Exception:
            # 已持久事件不能撒谎称“未发生”；该副本必须标记不可用并离线重建。
            self.up=False
            raise

    def create(self,payload:dict) -> dict:
        record=make_record(self.ledger,'create',payload)
        self.accept(record)
        return record

    def token(self,payload:dict) -> dict:
        record=make_record(self.ledger,'token',payload)
        self.accept(record)
        return record

    def recover_cache(self) -> dict:
        require(artifact_identity(self.tok,self.models)==self.identity,
                '恢复实例权重身份不匹配')
        details=self.journal.recover()
        # 丢弃所有易失KV，依据逻辑会话从最早提示重建。
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(self.pool.capacity_blocks,self.pool.block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(self.tok,self.models,self.pool)
        total=0;active=0
        for rid,state in sorted(self.ledger.sessions.items()):
            if state.terminal is not None:continue
            active+=1
            self.runner.open_prefill(rid,state.version,state.prompt_ids)
            total+=len(state.prompt_ids)
            for t in state.generated:
                require(self._expected(rid)==t,
                        '恢复后同一前缀的贪心输出不同，拒绝身份/模型漂移')
                self.runner.step(rid,t)
                total+=1
            self.runner.assert_consistent(rid)
        self.up=True
        details.update({'active_sessions_rebuilt':active,'tokens_recomputed':total,
                        'kv_tensors_restored_from_disk':False,'kv_live_blocks':self.pool.usage()['live_blocks']})
        self.last_recovery=details
        return details

    def audit(self):
        active={k for k,v in self.ledger.sessions.items() if v.terminal is None}
        require(set(self.pool.sessions)==active and set(self.runner.last_logits)==active,
                '逻辑活跃会话与KV实体不一致')
        max_diff=0.0
        for rid in sorted(active):
            st=self.ledger.sessions[rid]
            require(self.pool.sessions[rid].tokens==st.prompt_ids+st.generated,
                    'KV与已确认的逻辑token历史不一致')
            require(self.pool.sessions[rid].version==st.version,'KV与持久版本绑定不一致')
            max_diff=max(max_diff,self.runner.assert_consistent(rid))
        u=self.pool.audit()
        return {'replica':self.name,'logical_seq':self.ledger.seq,'logical_requests':len(self.ledger.sessions),
                'active_sessions':len(active),'kv_live_blocks':u['live_blocks'],
                'max_cache_full_logits_diff':max_diff,'models':self.identity['weights']}


def _request_payload(tok,tenant,key,version,text,max_new_tokens):
    rid=make_rid(tenant,key)
    require(version in {'stable','canary'},'模型版本非法')
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=4,'输出预算非法')
    prompt=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                   add_generation_prompt=True)
    return {'rid':rid,'tenant':tenant,'key':key,'version':version,
            'prompt_ids':prompt,'max_new_tokens':max_new_tokens}

In [9]:
# %% 07 单控制点epoch围栏、同步/异步确认与接管

class SimulatedCrash(RuntimeError):
    """明确模拟写盘之后、客户端收到ACK之前的中断，不执行真实进程杀死。"""


def save_model_artifacts(path:Path,tok,models) -> dict:
    require(not Path(path).exists(),'模型artifact不可覆盖')
    identity=artifact_identity(tok,models)
    blob={'version':VERSION,'identity':identity,'tokenizer':tok.contract(),
          'weights':{version:{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                     for version,model in models.items()}}
    torch.save(blob,path)
    return identity


def load_model_artifacts(path:Path):
    # 仅加载本课自己生成、可信来源的纯张量与dict；不执行外部网络下载。
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'artifact版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer/chat template身份不匹配')
    identity=blob['identity']
    cfg=ToyConfig(**identity['config'])
    models={}
    for name,values in blob['weights'].items():
        model=inject_lora(TinyChatLM(cfg),rank=4,alpha=8.0)
        model.load_state_dict(values,strict=True)
        models[name]=model.eval().requires_grad_(False)
    require(artifact_identity(tok,models)==identity,'模型artifact哈希不匹配')
    return tok,models


class Coordinator:
    """一个教学用串行控制器，持久化epoch与ACK审计；不是分布式共识。"""
    def __init__(self,root:Path,tok,models,policy='sync',seed=8101,
                 capacity_blocks=64,load_existing=False):
        require(policy in {'sync','async'},'确认模式只能sync/async')
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.control_path=self.root/'control.json'
        self.artifact_path=self.root/'models.pt'
        self.tok=tok;self.models=models
        self.model_identity=artifact_identity(tok,models)
        self.capacity_blocks=capacity_blocks
        self.seed=seed
        if load_existing:
            self.control=self._read_control()
            require(self.control['model_identity']==self.model_identity,
                    '重载模型与原先已确认事件的权重/词表身份不符')
            require(self.control['policy']==policy,'控制器确认策略不可暗中更改')
        else:
            require(not self.control_path.exists() and not self.artifact_path.exists(),
                    '根目录已包含集群持久状态，禁止覆盖')
            save_model_artifacts(self.artifact_path,tok,models)
            self.control={'version':VERSION,'active':'A','epoch':1,'policy':policy,
                          'model_identity':self.model_identity,'ack_seq':0,'ack_tick':0,
                          'ack_history':[],'failovers':[],'quarantined':[],'seed':seed,
                          'capacity_blocks':capacity_blocks}
            atomic_json(self.control_path,self.control)
        self.rep={name:Replica(name,self.root/name,tok,models,capacity_blocks,
                               load_existing=load_existing) for name in ('A','B')}
        self.updates_acknowledged=0
        for name in self.control['quarantined']:
            require(name!=self.control['active'],'活跃主不能被隔离')
            self.rep[name].shutdown()
        self._check_artifacts()

    def _read_control(self):
        try:state=json.loads(self.control_path.read_text(encoding='utf-8'))
        except (UnicodeDecodeError,json.JSONDecodeError,FileNotFoundError) as exc:
            raise LessonError('控制指针无法读取') from exc
        require(type(state) is dict and
                set(state)=={'version','active','epoch','policy','model_identity','ack_seq',
                             'ack_tick','ack_history','failovers','quarantined','seed','capacity_blocks'},
                '控制记录字段非法')
        require(state['version']==VERSION and state['active'] in {'A','B'} and
                type(state['epoch']) is int and state['epoch']>=1,'控制代际非法')
        require(type(state['ack_seq']) is int and state['ack_seq']>=0 and
                state['ack_tick']==state['ack_seq'],'ACK水位非法')
        require(type(state['quarantined']) is list and
                set(state['quarantined'])<={'A','B'} and
                state['active'] not in state['quarantined'],
                '隔离副本列表非法')
        return state

    def _check_artifacts(self):
        token,models=load_model_artifacts(self.artifact_path)
        require(artifact_identity(token,models)==self.model_identity,
                '模型artifact与当前控制身份不符')
        for r in self.rep.values():
            require(r.identity==self.model_identity,'副本模型身份漂移')
            require(artifact_identity(r.tok,r.models)==r.identity,'运行中模型权重被改写')

    def _fence(self,epoch:int):
        durable=self._read_control()
        require(type(epoch) is int and epoch==durable['epoch'] and
                durable['active']==self.control['active'] and
                durable['epoch']==self.control['epoch'],
                'epoch过期或控制指针已被其他控制器切换；拒绝陈旧主节点写入')
        require(self.rep[durable['active']].up,'活跃副本不可用')
        return self.rep[durable['active']]

    def _standby(self):return self.rep['B' if self.control['active']=='A' else 'A']

    def replicate_pending(self):
        primary=self.rep[self.control['active']];target=self._standby()
        require(target.name not in self.control['quarantined'],
                '备机日志已隔离，必须显式丢弃旧分叉并重建')
        require(target.up,'备机不可用，不能确认同步复制')
        require(target.identity==primary.identity,'复制副本模型身份不相等')
        left=target.journal.records;right=primary.journal.records
        require(len(left)<=len(right) and
                all(a==b for a,b in zip(left,right)),
                '日志发生分叉：不可自动拼接，必须隔离并人工处理')
        count=0
        for row in right[len(left):]:
            target.accept(row);count+=1
        require(target.ledger.last_hash==primary.journal.records[target.ledger.seq-1]['hash']
                if target.ledger.seq else target.ledger.last_hash==ZERO_HASH,
                '复制完成后head hash不一致')
        return count

    def _ack(self,row:dict):
        require(type(row) is dict and 'seq' in row and 'hash' in row,'ACK没有WAL证据')
        primary=self.rep[self.control['active']]
        require(primary.ledger.seq>=row['seq'] and
                primary.journal.records[row['seq']-1]==row,'主机日志和ACK不匹配')
        already_acked=row['seq']<=self.control['ack_seq']
        if already_acked:
            require(any(e['seq']==row['seq'] and e['hash']==row['hash']
                        for e in self.control['ack_history']),
                    '历史ACK证据与当前日志不匹配')
        elif self.control['policy']=='sync':
            standby=self._standby()
            require(standby.name not in self.control['quarantined'] and
                    standby.up and standby.ledger.seq>=row['seq'] and
                    standby.journal.records[row['seq']-1]==row,'同步ACK前备机没有持久化该事件')
        if row['seq']>self.control['ack_seq']:
            require(row['seq']==self.control['ack_seq']+1,'不能越过未确认的先前事件')
            self.control['ack_seq']=row['seq'];self.control['ack_tick']=row['tick']
            self.control['ack_history'].append({'seq':row['seq'],'hash':row['hash'],
                                               'epoch':self.control['epoch']})
            atomic_json(self.control_path,self.control)
            self.updates_acknowledged+=1
        return {'seq':row['seq'],'hash':row['hash'],'epoch':self.control['epoch'],
                'durability':'primary+standby' if self.control['policy']=='sync' else 'primary_only'}

    def _commit(self,row:dict,crash_after_durable=False):
        primary=self.rep[self.control['active']]
        primary.accept(row)   # 先写盘fsync，再响应或复制。
        if self.control['policy']=='sync':self.replicate_pending()
        if crash_after_durable:
            raise SimulatedCrash('WAL已fsync但控制ACK未写入；客户端必须同键对账重试')
        return self._ack(row)

    def start(self,epoch,tenant,key,version='stable',text='hello',max_new_tokens=4,
              crash_after_durable=False):
        active=self._fence(epoch)
        payload=_request_payload(self.tok,tenant,key,version,text,max_new_tokens)
        rid=payload['rid']
        if rid in active.ledger.sessions:
            item=active.ledger.sessions[rid]
            require(item.identity()=={k:payload[k] for k in item.identity()},
                    '同一幂等键与先前请求内容/版本不一致')
            row=next(r for r in active.journal.records if r['kind']=='create'
                     and r['payload']['rid']==rid)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'new_event':False,'ack':ack}
        row=make_record(active.ledger,'create',payload)
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'new_event':True,'ack':ack}

    def step(self,epoch,rid,expected_index:int,crash_after_durable=False):
        active=self._fence(epoch)
        require(rid in active.ledger.sessions,'请求不存在')
        item=active.ledger.sessions[rid]
        require(type(expected_index) is int and expected_index>=0,'续传序号非法')
        if expected_index<len(item.generated):
            row=next(r for r in active.journal.records if r['kind']=='token' and
                     r['payload']['rid']==rid and r['payload']['index']==expected_index)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'index':expected_index,'token':row['payload']['token'],
                    'terminal':row['payload']['terminal'],'new_event':False,'ack':ack}
        require(expected_index==len(item.generated),'不能跳过尚未产生的token')
        require(item.terminal is None,'请求已经终止')
        tok=active._expected(rid)
        stop='eos' if tok==self.tok.eos_token_id else (
             'length' if expected_index+1==item.max_new_tokens else None)
        row=make_record(active.ledger,'token',{'rid':rid,'index':expected_index,
                                              'token':tok,'terminal':stop})
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'index':expected_index,'token':tok,'terminal':stop,
                'new_event':True,'ack':ack}

    def checkpoint(self):
        return {name:self.rep[name].journal.checkpoint() for name in ('A','B')}

    def promote(self,epoch,target='B',allow_data_loss=False,rpo_budget_ticks=0,
                rto_budget_ticks=12,tokens_per_tick=32,wal_records_per_tick=16):
        require(type(allow_data_loss) is bool,'数据丢失授权必须显式bool')
        require(target in self.rep and target!=self.control['active'],'接管目标非法')
        persisted=self._read_control()
        require(type(epoch) is int and epoch==self.control['epoch'] and
                persisted==self.control, '接管用epoch过期或控制状态发生变化')
        # 故障后原主机可down；只有这个教学控制指针继续提供围栏，不由原主授权。
        require(all(type(v) is int and v>=0 for v in (rpo_budget_ticks,rto_budget_ticks))
                and tokens_per_tick>0 and wal_records_per_tick>0,'恢复目标/预算非法')
        old_active=self.control['active'];standby=self.rep[target]
        require(standby.up,'备机不可用')
        self._check_artifacts()
        # 在新对象中只从本地持久日志及快照恢复，易失KV不复制过来。
        fresh=Replica(target,self.root/target,self.tok,self.models,self.capacity_blocks,
                      load_existing=True)
        require(artifact_identity(fresh.tok,fresh.models)==self.model_identity,
                '灾难恢复模型/模板指纹不一致')
        audit=fresh.audit()
        ack_seq=self.control['ack_seq']
        lost=max(0,ack_seq-fresh.ledger.seq)
        require(fresh.ledger.seq<=self.rep[old_active].ledger.seq,
                '备机不能凭空比主机多出旧epoch事件')
        # 完整一致的确认前缀不能只是seq相等；还要比较哈希链前缀。
        primary_records=self.rep[old_active].journal.records
        require(fresh.journal.records==primary_records[:len(fresh.journal.records)],
                '备机历史并非旧主已提交前缀；禁止接管分叉日志')
        rpo_gap=max(0,self.control['ack_tick']-fresh.ledger.tick)
        rto=2+math.ceil(fresh.last_recovery['wal_records_verified']/wal_records_per_tick)+(
             math.ceil(fresh.last_recovery['tokens_recomputed']/tokens_per_tick))
        if (lost>0 or rpo_gap>rpo_budget_ticks) and not allow_data_loss:
            raise LessonError('未复制的已ACK事件存在，自动接管必须阻断并报告RPO风险')
        if rto>rto_budget_ticks:
            raise LessonError('预计恢复逻辑tick超出RTO门槛；本课拒绝自动接管')
        report={'from':old_active,'to':target,'old_epoch':epoch,'new_epoch':epoch+1,
                'sync_mode':self.control['policy'],'ack_seq_before':ack_seq,
                'target_seq':fresh.ledger.seq,'lost_acked_events':lost,
                'rpo_gap_logical_ticks':rpo_gap,'rpo_objective_ticks':rpo_budget_ticks,
                'rto_simulated_ticks':rto,'rto_objective_ticks':rto_budget_ticks,
                'rto_formula':'2 + ceil(WAL_verified / records_per_tick) + ceil(KV_replay_tokens / tokens_per_tick)',
                'recovered_active_sessions':audit['active_sessions'],
                'recovered_tokens_recomputed':fresh.last_recovery['tokens_recomputed'],
                'data_loss_override':allow_data_loss and lost>0,
                'no_real_time_measurement':True}
        new_control=copy.deepcopy(self.control)
        new_control['active']=target;new_control['epoch']=epoch+1
        new_control['ack_seq']=fresh.ledger.seq;new_control['ack_tick']=fresh.ledger.tick
        if lost>0:
            new_control['ack_history']=[row for row in new_control['ack_history']
                                        if row['seq']<=fresh.ledger.seq]
            new_control['quarantined']=sorted(set(new_control['quarantined'])|{old_active})
        new_control['failovers'].append(copy.deepcopy(report))
        # 本地控制指针fsync后再关闭旧主；失效旧epoch依赖这个唯一可信控制点。
        atomic_json(self.control_path,new_control)
        self.rep[old_active].shutdown()
        standby.shutdown()
        self.rep[target]=fresh
        self.control=new_control
        return report

    def audit(self):
        require(self._read_control()==self.control,'控制内存与持久状态不一致')
        self._check_artifacts()
        active=self.rep[self.control['active']]
        require(active.up,'当前活跃副本不可用')
        return {'active':self.control['active'],'epoch':self.control['epoch'],
                'ack_seq':self.control['ack_seq'], 'artifacts':self.model_identity,
                'active_audit':active.audit(),
                'standby_head':self._standby().ledger.seq}


def _replica_shutdown(self):
    for sid in list(self.pool.sessions):
        if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
        self.runner.close(sid)
    require(self.pool.audit()['live_blocks']==0,'关闭副本仍有物理KV泄漏')
    self.up=False

Replica.shutdown=_replica_shutdown


def restore_controller(root:Path) -> Coordinator:
    root=Path(root)
    tok,models=load_model_artifacts(root/'models.pt')
    control=json.loads((root/'control.json').read_text(encoding='utf-8'))
    result=Coordinator(root,tok,models,policy=control['policy'],seed=control['seed'],
                       capacity_blocks=control['capacity_blocks'],load_existing=True)
    result.audit()
    return result

## 4. 预留容量必须覆盖活跃请求的未来长度

只看“现在用了几页”会低估 failover 恢复负担。保守估计应对每个活跃请求分别计算未来最大长度：

\[
B=\sum_i \left\lceil\frac{L_i+\mathrm{remaining}_i}{\mathrm{block\ size}}\right\rceil
+\sum_j\left\lceil\frac{L_j^{new}}{\mathrm{block\ size}}\right\rceil
+ B_{reserve}
\]

每个会话**先向上取整**，不是把所有长度相加后再取整。本课不将前缀共享的潜在节省算进最低容量保证。对新请求，预先加上它的 prompt 与最大生成长度预算。

这只计算物理 KV 页，不包括模型权重、临时张量、CUDA allocator 碎片或分布式资源预留，因此不能照搬为显存预算。

In [10]:
# %% 08 保守的KV容量上界（每会话单独向上取整）
def capacity_projection(replica:Replica,available_blocks:int,reserve_blocks:int,
                        planned_new_session_lengths:list[int]|None=None):
    """每会话保守按未来最长上下文估算页数；不计共享前缀收益，避免把优化当容量保证。"""
    require(type(available_blocks) is int and available_blocks>0 and
            type(reserve_blocks) is int and reserve_blocks>=0,'容量必须非负整数且池大小为正')
    new=[] if planned_new_session_lengths is None else planned_new_session_lengths
    require(type(new) is list and all(type(n) is int and 0<n<=replica.pool.cfg.max_length for n in new),
            '新请求长度预算必须为正且不超过上下文')
    block=replica.pool.block_size
    current=0;worst=0;active=0
    for st in replica.ledger.sessions.values():
        if st.terminal is not None:continue
        active+=1
        length=len(st.prompt_ids)+len(st.generated)
        remaining=st.max_new_tokens-len(st.generated)
        require(0<=remaining<=4 and length+remaining<=replica.pool.cfg.max_length,
                '会话剩余token预算不合法')
        current+=math.ceil(length/block)
        worst+=math.ceil((length+remaining)/block)
    worst+=sum(math.ceil(x/block) for x in new)
    total=worst+reserve_blocks
    bytes_per_block=replica.pool.bytes_per_block
    return {'active_sessions':active,'current_required_blocks':current,
            'projected_session_blocks':worst,'emergency_reserved_blocks':reserve_blocks,
            'total_required_blocks':total,'available_blocks':available_blocks,
            'headroom_blocks':available_blocks-total,'allowed':available_blocks>=total,
            'bytes_per_block':bytes_per_block,'projected_reserved_bytes':total*bytes_per_block,
            'physical_capacity_bytes':available_blocks*bytes_per_block,
            'model':'per-session conservative; no prefix share credit; live KV only; excludes model weights'}

## 5. 配置 schema 不允许隐式忽略未知字段

`ServingConfig` 明确记录：

| 类别 | 本课字段 | 处理方式 |
|---|---|---|
| 运行身份 | 模型/模板指纹、同步 WAL | 本次不能只通过配置切换假冒更新 |
| 路由 | canary%、确定性 salt | 受控变更；已存在请求从 WAL 固定版本 |
| 隔离 | 已批准租户白名单 | 有活跃会话时不能突然撤销该租户 |
| 容量 | 最大会话数、KV最低保留页数 | 双副本逐一验证 |
| 控制 | 新请求准入开关 | 显式 bool；不能把 `1` 冒充 `True` |
| 观测 | telemetry 抽样比例 | 即使只改观测字段，漂移也必须可见 |

不接受未知字段、漏字段、未排序/重复租户、布尔值冒充数值。漂移检查比较明确的实际运行态字段，不能只根据文本差异或文件时间戳判断。

In [11]:
# %% 09 声明式配置：意图、身份与运行态是三个不同事实
@dataclass(frozen=True)
class ServingConfig:
    schema: str
    model_identity_hash: str
    tokenizer_template_hash: str
    wal_policy: str
    route_canary_percent: int
    routing_salt: str
    allowed_tenants: tuple[str, ...]
    max_active_sessions: int
    reserve_blocks: int
    admission_enabled: bool
    telemetry_sample_percent: int

    def validate(self):
        require(self.schema == 'serving-config-v1', '未知配置模式，不能忽略字段')
        for label, value in (('模型身份', self.model_identity_hash), ('分词模板', self.tokenizer_template_hash)):
            require(type(value) is str and bool(re.fullmatch('[0-9a-f]{64}', value)), label+'哈希非法')
        require(self.wal_policy == 'sync', '本课零RPO发布只支持同步持久化ACK')
        require(type(self.route_canary_percent) is int and 0 <= self.route_canary_percent <= 100,
                'canary比例必须为整数0..100')
        require(type(self.telemetry_sample_percent) is int and 0 <= self.telemetry_sample_percent <= 100,
                '观测抽样比例必须为整数0..100')
        require(type(self.routing_salt) is str and bool(re.fullmatch('[A-Za-z0-9_-]{1,24}', self.routing_salt)),
                '路由salt非法')
        require(type(self.allowed_tenants) is tuple and 0 < len(self.allowed_tenants) <= 8 and
                all(type(t) is str and bool(re.fullmatch('[a-z][a-z0-9_-]{0,15}',t)) for t in self.allowed_tenants) and
                len(set(self.allowed_tenants)) == len(self.allowed_tenants) and
                tuple(sorted(self.allowed_tenants)) == self.allowed_tenants,
                '租户列表必须排序、唯一且受限')
        require(type(self.max_active_sessions) is int and 1 <= self.max_active_sessions <= 16,
                '会话上限非法')
        require(type(self.reserve_blocks) is int and 2 <= self.reserve_blocks <= 128,
                '物理KV应保留至少两页紧急余量')
        require(type(self.admission_enabled) is bool, '准入开关必须是显式bool')

    def wire(self) -> dict:
        self.validate()
        data=asdict(self);data['allowed_tenants']=list(self.allowed_tenants)
        return data

    @staticmethod
    def from_wire(data:dict):
        require(type(data) is dict and set(data)==set(ServingConfig.__dataclass_fields__),
                '配置字段不完整/有未知字段，不进行隐式降级')
        require(type(data['allowed_tenants']) is list, 'JSON中的租户必须是列表')
        obj=ServingConfig(**{**data,'allowed_tenants':tuple(data['allowed_tenants'])})
        obj.validate()
        require(obj.wire()==data,'配置无法无损往返')
        return obj


def initial_serving_config(cluster:Coordinator) -> ServingConfig:
    identity=artifact_identity(cluster.tok,cluster.models)
    result=ServingConfig('serving-config-v1',digest(identity),identity['tokenizer'],'sync',
                         0,'course83',('alpha','beta'),8,4,True,10)
    result.validate();return result


def changed_config(cfg:ServingConfig, **changes) -> ServingConfig:
    data=cfg.wire();data.update(changes)
    result=ServingConfig.from_wire(data)
    return result


def config_diff(before:ServingConfig,after:ServingConfig) -> list[dict]:
    a,b=before.wire(),after.wire()
    return [{'path':key,'old':a[key],'new':b[key],
             'severity':'critical' if key in {'model_identity_hash','tokenizer_template_hash',
               'wal_policy','allowed_tenants','max_active_sessions','reserve_blocks','admission_enabled'}
               else 'controlled'} for key in sorted(a) if a[key]!=b[key]]

In [12]:
demo_spec = ServingConfig('serving-config-v1', 'a'*64,'b'*64,'sync',0,'demo',('alpha','beta'),8,4,True,10)
print('schema往返:', ServingConfig.from_wire(demo_spec.wire()) == demo_spec)
proposed_spec = changed_config(demo_spec, route_canary_percent=20, telemetry_sample_percent=30)
print('实际配置差异:', json.dumps(config_diff(demo_spec, proposed_spec), ensure_ascii=False, indent=2))
for broken in [{'route_canary_percent':True}, {'allowed_tenants':['beta','alpha']}, {'reserve_blocks':0}]:
    try:
        changed_config(demo_spec, **broken)
    except LessonError as error:
        print('拒绝非法变更:', error)

schema往返: True
实际配置差异: [
  {
    "path": "route_canary_percent",
    "old": 0,
    "new": 20,
    "severity": "controlled"
  },
  {
    "path": "telemetry_sample_percent",
    "old": 10,
    "new": 30,
    "severity": "controlled"
  }
]
拒绝非法变更: canary比例必须为整数0..100
拒绝非法变更: 租户列表必须排序、唯一且受限
拒绝非法变更: 物理KV应保留至少两页紧急余量


## 6. 发布流程：双副本预置，再切换唯一活动指针

```text
期望候选（不可直接当已生效）
   ↓ schema、模型/模板身份、WAL模式校验
   ↓ 真实CPU KV探针与两副本容量检查
   ↓ 绑定旧配置代际、ACK水位、epoch、TTL的预检证据
   ↓ 校验模拟审批记录与预期generation
   ↓ 只预置B和A副本下一代配置（不生效）
   ↓ 再核对实际状态没有变化
   ↓ 原子切换单个本地活动指针
   ↓ 记录可核查审计，再允许新流量
```

第一个副本预置完成时若进程失败，**旧活动指针仍然有效**。如果指针已经提交而客户端没收到ACK，则必须读取真实指针并做幂等对账，不是无脑再申请新版本。

这只是**单进程串行 CAS 的教材**。真实多写入方必须使用可信控制服务与代际/资源版本机制。Kubernetes Server-Side Apply 对字段所有权冲突有更完善的管理。[Kubernetes 官方文档](https://kubernetes.io/docs/reference/using-api/server-side-apply/)

In [13]:
# %% 10 单控制点版本清单、双副本预置与原子活动指针
class ChangeLedger:
    """本地hash链+单独封存head；不能证明外部主体身份或防恶意重写两者。"""
    def __init__(self,path:Path,load_existing=False):
        self.path=Path(path);self.head_path=self.path.with_suffix('.head.json')
        if load_existing:
            self.verify();return
        require(not self.path.exists() and not self.head_path.exists(),'不能覆盖历史变更审计')
        self.path.parent.mkdir(parents=True,exist_ok=True)
        self.path.touch()
        atomic_json(self.head_path,{'count':0,'head':ZERO_HASH})

    def verify(self) -> dict:
        try: lines=self.path.read_bytes().splitlines();head=json.loads(self.head_path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('变更审计缺失/损坏') from exc
        require(type(head) is dict and set(head)=={'count','head'},'审计head字段非法')
        last=ZERO_HASH;records=[]
        for i,raw in enumerate(lines,1):
            try:record=json.loads(raw.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('审计JSON行损坏') from exc
            require(type(record) is dict and set(record)=={'seq','tick','event','data','prev','hash'},
                    '变更审计字段不匹配')
            require(type(record['seq']) is int and record['seq']==i and record['prev']==last,
                    '审计事件序号/前向链错误')
            require(type(record['tick']) is int and 0<=record['tick']<=100000,
                    '逻辑tick非法')
            require(type(record['event']) is str and bool(record['event']) and
                    type(record['data']) is dict,'审计事件非法')
            core={k:v for k,v in record.items() if k!='hash'}
            require(record['hash']==digest(core),'审计内容被改写')
            last=record['hash'];records.append(record)
        require(head=={'count':len(records),'head':last},'审计内容与封存head不一致')
        return {'count':len(records),'head':last,'events':[r['event'] for r in records]}

    def add(self,tick:int,event:str,data:dict) -> dict:
        previous=self.verify()
        require(type(tick) is int and 0<=tick<=100000,'tick非法')
        require(type(event) is str and bool(re.fullmatch('[a-z_]{2,48}',event)),'事件名非法')
        require(type(data) is dict and len(canonical(data).encode())<=2048,
                '审计字段过大：不允许写入原始prompt、token和密钥')
        require(not any('prompt' in k.lower() or 'secret' in k.lower() or 'token_ids' in k.lower()
                        for k in data),'原始prompt/token不能写入审计')
        last=self.path.read_bytes().splitlines()
        if last:
            previous_record=json.loads(last[-1]);require(tick>=previous_record['tick'],'审计tick不能倒退')
        core={'seq':previous['count']+1,'tick':tick,'event':event,
              'data':copy.deepcopy(data),'prev':previous['head']}
        row={**core,'hash':digest(core)}
        with self.path.open('ab') as f:
            f.write((canonical(row)+'\n').encode('utf-8'));f.flush();os.fsync(f.fileno())
        atomic_json(self.head_path,{'count':row['seq'],'head':row['hash']})
        self.verify();return row


class ConfigStore:
    """单进程串行CAS实验，不能替代K8s apiserver/etcd/租约或真正分布式共识。"""
    def __init__(self,root:Path,cluster:Coordinator,load_existing=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.cluster=cluster
        self.ptr=self.root/'active.json';self.versions=self.root/'versions'
        self.applied=self.root/'replicas';self.audit=ChangeLedger(self.root/'changes.jsonl',load_existing)
        self.versions.mkdir(exist_ok=True)
        for name in ('A','B'):(self.applied/name).mkdir(parents=True,exist_ok=True)
        if load_existing:
            self.read_active(require_healthy=True)
        else:
            require(not self.ptr.exists(),'活动指针不可覆盖')
            initial=initial_serving_config(cluster)
            self._stage(1,initial)
            atomic_json(self.ptr,self._pointer(1,initial))
            self.audit.add(0,'initial_release',{'generation':1,'config_sha':digest(initial.wire())})
            self.read_active(require_healthy=True)

    @staticmethod
    def _pointer(generation:int, config:ServingConfig) -> dict:
        return {'generation':generation,'sha':digest(config.wire())}

    def _version_path(self,generation:int) -> Path:
        require(type(generation) is int and generation>=1,'配置代际非法')
        return self.versions/f'{generation:06d}.json'

    def _replica_path(self,name:str,generation:int) -> Path:
        require(name in {'A','B'},'副本非法')
        return self.applied/name/f'{generation:06d}.json'

    def read_pointer(self) -> dict:
        try:state=json.loads(self.ptr.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('活动版本指针不可读取') from exc
        require(type(state) is dict and set(state)=={'generation','sha'},'活动指针字段不对')
        require(type(state['generation']) is int and state['generation']>=1 and
                type(state['sha']) is str and bool(re.fullmatch('[0-9a-f]{64}',state['sha'])),
                '活动指针值非法')
        return state

    def _load_config(self,path:Path) -> ServingConfig:
        try:content=json.loads(path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('副本或版本配置损坏') from exc
        return ServingConfig.from_wire(content)

    def read_active(self,require_healthy=True) -> tuple[dict,ServingConfig]:
        p=self.read_pointer();spec=self._load_config(self._version_path(p['generation']))
        require(digest(spec.wire())==p['sha'],'活动指针与被选版本哈希不一致')
        if require_healthy:self.assert_aligned(p,spec)
        return p,spec

    def assert_aligned(self,p:dict|None=None,spec:ServingConfig|None=None) -> dict:
        if p is None or spec is None:p,spec=self.read_active(False)
        observed={}
        for name in ('A','B'):
            got=self._load_config(self._replica_path(name,p['generation']))
            require(got.wire()==spec.wire(),f'{name}副本配置漂移；停止新请求与自动接管')
            observed[name]=digest(got.wire())
        model_identity=artifact_identity(self.cluster.tok,self.cluster.models)
        require(spec.model_identity_hash==digest(model_identity) and
                spec.tokenizer_template_hash==model_identity['tokenizer'],
                '运行模型或模板身份与配置不一致')
        require(self.cluster.control['policy']==spec.wal_policy,'WAL确认模式发生漂移')
        require(self.cluster._read_control()==self.cluster.control,'控制指针与内存不一致')
        require(self.cluster.model_identity==model_identity,'实际模型权重身份变化')
        require(self.cluster.capacity_blocks>=spec.reserve_blocks,'物理KV页容量不足')
        return {'generation':p['generation'],'sha':p['sha'],'replicas':observed,
                'epoch':self.cluster.control['epoch'],'model_identity_hash':digest(model_identity)}

    def _immutable_json(self,path:Path,content:dict):
        if path.exists():
            existing=self._load_config(path)
            require(existing.wire()==content,'预置文件已存在但内容不同：不能自动覆盖')
        else:
            atomic_json(path,content)

    def _stage(self,generation:int,cfg:ServingConfig,fail_after:str|None=None):
        data=cfg.wire()
        self._immutable_json(self._version_path(generation),data)
        for name in ('A','B'):
            self._immutable_json(self._replica_path(name,generation),data)
            if fail_after=='stage_'+name:
                raise SimulatedCrash('只预置了部分副本：活动指针未改变，不能宣称发布成功')
        # 预置阶段的文件只能通过指针切换变为生效配置。
        for name in ('A','B'):
            require(self._load_config(self._replica_path(name,generation)).wire()==data,
                    '预置副本内容不匹配')

    def snapshot(self) -> dict:
        p,s=self.read_active(require_healthy=True)
        control=self.cluster.control
        return {'pointer':copy.deepcopy(p),'config_sha':digest(s.wire()),
                'epoch':control['epoch'],'active':control['active'],
                'ack_seq':control['ack_seq'],'control_sha':digest(control),
                'A_seq':self.cluster.rep['A'].ledger.seq,
                'B_seq':self.cluster.rep['B'].ledger.seq,
                'artifact_sha':digest(self.cluster.model_identity)}

    def probe_kv(self) -> dict:
        self.assert_aligned()
        reports={};max_delta=0.0
        tok=self.cluster.tok
        prompt=tok.apply_chat_template([{'role':'user','content':'hello'}],
                                       tokenize=True,add_generation_prompt=True)
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            require(replica.up,'副本未运行，发布预检拒绝')
            require(artifact_identity(tok,replica.models)==self.cluster.model_identity,
                    '副本实际模型权重身份与发布不匹配')
            for version in ('stable','canary'):
                pool=PagedKVPool(32,4,next(iter(replica.models.values())).cfg,set(replica.models))
                runner=CachedModelRunner(tok,replica.models,pool)
                sid='preflight_'+name+'_'+version
                runner.open_prefill(sid,version,prompt)
                delta=runner.assert_consistent(sid)
                max_delta=max(max_delta,delta)
                runner.close(sid)
                require(pool.audit()['live_blocks']==0,'preflight的物理KV没有释放')
                reports[name+'_'+version]={'max_cache_full_diff':delta,'released':True}
        return {'max_cache_full_diff':max_delta,'checked':reports,'real_tensor_kv':True}

    def safe_capacity(self,proposed:ServingConfig,planned_new_session_lengths:list[int]|None=None) -> dict:
        proposed.validate()
        result={}
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            check=capacity_projection(replica,self.cluster.capacity_blocks,proposed.reserve_blocks,
                                      planned_new_session_lengths)
            require(check['allowed'],f'{name}可恢复KV容量不够；不准上线')
            require(check['active_sessions']<=proposed.max_active_sessions,
                    f'{name}现有会话数超过新配置上限')
            result[name]=check
        return result

    def preflight(self,candidate:ServingConfig,tick:int,ttl_ticks:int=4) -> dict:
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '使用非负的教学逻辑tick；TTL为1..20')
        candidate.validate();p,old=self.read_active(require_healthy=True)
        require(candidate.model_identity_hash==old.model_identity_hash and
                candidate.tokenizer_template_hash==old.tokenizer_template_hash and
                candidate.wal_policy==old.wal_policy,
                '模型/模板/WAL身份变更需要独立的新artifact迁移，不能只修改配置字段')
        for name in ('A','B'):
            for st in self.cluster.rep[name].ledger.sessions.values():
                require(st.terminal is not None or st.tenant in candidate.allowed_tenants,
                        '存在活跃会话时不能撤销该租户访问权限')
        checks=self.safe_capacity(candidate)
        probes=self.probe_kv()
        snap=self.snapshot()
        report={'kind':'release_preflight','base':snap,'candidate':candidate.wire(),
                'candidate_sha':digest(candidate.wire()),'issued_tick':tick,'expires_tick':tick+ttl_ticks,
                'capacity':checks,'probe':probes,'diff':config_diff(old,candidate),
                'decision':'allow','limitations':'logical_ticks_single_controller_no_authenticated_approval'}
        report['seal']=digest(report)
        return report

    def verify_preflight(self,proof:dict,candidate:ServingConfig,tick:int) -> None:
        require(type(proof) is dict and 'seal' in proof and
                proof['seal']==digest({k:v for k,v in proof.items() if k!='seal'}),
                '发布证据完整性错误')
        require(proof['kind']=='release_preflight' and proof['decision']=='allow',
                '证据没有明确放行')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '预检证据尚未生效或已超过逻辑TTL')
        require(proof['candidate']==candidate.wire() and proof['candidate_sha']==digest(candidate.wire()),
                '预检证据绑定的是另一个候选')
        require(proof['base']==self.snapshot(),
                '预检后控制epoch/ACK/副本状态/生效配置改变，必须重新取证')
        self.safe_capacity(candidate)
        self.probe_kv()

    @staticmethod
    def approval(candidate:ServingConfig,base_generation:int,actor='local_operator') -> dict:
        return {'actor':actor,'ticket':'CHG-0083','approved':True,
                'candidate_sha':digest(candidate.wire()),'expected_generation':base_generation}

    def _approval(self,approve:dict,candidate:ServingConfig,expected_generation:int):
        require(type(approve) is dict and set(approve)==
                {'actor','ticket','approved','candidate_sha','expected_generation'},
                '审批记录字段不匹配')
        require(approve['approved'] is True and approve['expected_generation']==expected_generation and
                approve['candidate_sha']==digest(candidate.wire()) and
                type(approve['actor']) is str and bool(approve['actor']) and
                approve['ticket']=='CHG-0083','模拟审批没有绑定这次变更或代际')

    def publish(self,candidate:ServingConfig,proof:dict,approval:dict,
                expected_generation:int,tick:int,fail_after:str|None=None) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(type(expected_generation) is int and p['generation']==expected_generation,
                '配置CAS预期代际过期；拒绝并行覆盖')
        self._approval(approval,candidate,expected_generation)
        self.verify_preflight(proof,candidate,tick)
        generation=expected_generation+1
        self._stage(generation,candidate,fail_after)
        # 提交前再次检查；这只是单进程串行CAS模型，跨进程仍需真正的控制服务。
        require(self.read_pointer()==p,'预置期间活动指针变化，停止提交')
        self.verify_preflight(proof,candidate,tick)
        atomic_json(self.ptr,self._pointer(generation,candidate))
        if fail_after=='after_commit':
            raise SimulatedCrash('活动指针已切换但客户端未收到ACK：必须读取实际版本，不允许自动再次发布')
        self.assert_aligned()
        self.audit.add(tick,'release_committed',{'generation':generation,'candidate_sha':digest(candidate.wire()),
                                                'from_generation':expected_generation})
        return {'generation':generation,'active_sha':digest(candidate.wire()),
                'diff':proof['diff'],'ack':'committed','cas_expected':expected_generation}

    def detect_drift(self) -> dict:
        p,s=self.read_active(False)
        findings=[]
        for name in ('A','B'):
            try:observed=self._load_config(self._replica_path(name,p['generation']))
            except LessonError as exc:
                findings.append({'replica':name,'status':'invalid','reason':str(exc)})
                continue
            for change in config_diff(s,observed):
                findings.append({'replica':name,'status':'drift',**change})
        actual=artifact_identity(self.cluster.tok,self.cluster.models)
        if digest(actual)!=s.model_identity_hash:
            findings.append({'replica':'models','status':'drift','path':'model_identity_hash'})
        if self.cluster.control['policy']!=s.wal_policy:
            findings.append({'replica':'wal','status':'drift','path':'wal_policy'})
        return {'generation':p['generation'],'findings':findings,'healthy':not findings}

    def reconcile(self,name:str,approval:dict,tick:int,expected_generation:int):
        require(name in {'A','B'},'副本名非法')
        p,s=self.read_active(False)
        require(p['generation']==expected_generation,'修复预期代际过期')
        self._approval(approval,s,expected_generation)
        self._load_config(self._version_path(p['generation']))
        # 单控制者的显式修复动作，不掩盖WAL/真实权重漂移，不自动删除历史。
        path=self._replica_path(name,p['generation'])
        before=self.detect_drift()['findings']
        require(any(item['replica']==name for item in before),'未检测到该副本漂移，拒绝无意义覆盖')
        atomic_json(path,s.wire())
        self.assert_aligned()
        self.audit.add(tick,'drift_reconciled',{'generation':expected_generation,
                                                'replica':name,'restored_sha':digest(s.wire())})
        return {'reconciled':name,'generation':expected_generation,'healthy':True}

    def _has_commit_receipt(self,generation:int,sha:str) -> bool:
        self.audit.verify()
        for line in self.audit.path.read_text(encoding='utf-8').splitlines():
            record=json.loads(line)
            data=record['data']
            if record['event'] in {'initial_release','release_committed','release_reconciled'} and \
               data.get('generation')==generation and data.get('candidate_sha',data.get('config_sha'))==sha:
                return True
        return False

    def resolve_uncertain_commit(self,candidate:ServingConfig,proof:dict,approval:dict,
                                 expected_generation:int,tick:int) -> dict:
        """指针已经提交而ACK/审计未完成：读取真实指针，不盲目再递增版本。"""
        self._approval(approval,candidate,expected_generation)
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '无法核对原始预检证据')
        require(proof.get('candidate_sha')==digest(candidate.wire()) and
                proof.get('base',{}).get('pointer',{}).get('generation')==expected_generation,
                '含糊ACK绑定其他候选或前驱代际')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '含糊ACK过久，需要重新建立人工处置流程')
        p,s=self.read_active(require_healthy=True)
        require(p==self._pointer(expected_generation+1,candidate) and s.wire()==candidate.wire(),
                '未找到与原提交完全一致的活动指针；不能假装已经发布')
        require(self.cluster.control['epoch']==proof['base']['epoch'] and
                self.cluster.control['ack_seq']==proof['base']['ack_seq'] and
                self.cluster.control['active']==proof['base']['active'],
                '提交后出现新流量或切主，不能直接补写旧审计')
        recorded=self._has_commit_receipt(p['generation'],p['sha'])
        if not recorded:
            self.audit.add(tick,'release_reconciled',{'generation':p['generation'],
                              'candidate_sha':p['sha'],'from_generation':expected_generation})
        return {'generation':p['generation'],'committed':True,'audit_reconciled':not recorded,
                'pointer_advanced_again':False}

    def route(self,tenant:str,key:str,text:str='hello',max_new_tokens:int=4) -> dict:
        p,s=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置尚未有完整提交审计，先核对不确定ACK')
        require(s.admission_enabled,'当前配置停止接受新请求')
        require(tenant in s.allowed_tenants,'租户不在授权列表')
        active=self.cluster.rep[self.cluster.control['active']]
        require(active.up,'活动实例不可用')
        rid=make_rid(tenant,key)
        if rid in active.ledger.sessions:
            version=active.ledger.sessions[rid].version
        else:
            current=sum(st.terminal is None for st in active.ledger.sessions.values())
            require(current<s.max_active_sessions,'达到活动会话上限')
            future_prompt=self.cluster.tok.apply_chat_template(
                [{'role':'user','content':text}],tokenize=True,add_generation_prompt=True)
            self.safe_capacity(s,[len(future_prompt)+max_new_tokens])
            bucket=int(digest({'tenant':tenant,'key':key,'salt':s.routing_salt})[:12],16)%100
            version='canary' if bucket<s.route_canary_percent else 'stable'
        receipt=self.cluster.start(self.cluster.control['epoch'],tenant,key,version,text,max_new_tokens)
        return {**receipt,'version':version,'config_generation':p['generation'],
                'config_sha':p['sha'],'pinned_to_wal':True}

    def step(self,rid:str) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置缺少提交审计，停止生成')
        active=self.cluster.rep[self.cluster.control['active']]
        require(rid in active.ledger.sessions,'会话不存在')
        st=active.ledger.sessions[rid]
        require(st.terminal is None,'会话已经结束')
        return self.cluster.step(self.cluster.control['epoch'],rid,len(st.generated))

## 7. 发布预检为什么也会过期？

预检记录包括：

- 提交前 **活动配置版本与指纹**；
- 控制 epoch、已ACK日志序号、两个副本日志序号与模型身份；
- 候选配置的完整摘要，以及通过/拒绝结论；
- 双副本实际 KV logits 对照与容量余量；
- `issued_tick`、`expires_tick`、证据完整性摘要。

如果发放证据后又来了一个请求，ACK 水位变化，原证据就过时了，需要重新取证。逻辑 TTL 只用于可控模拟，不是现实世界的时间服务、电子签章或授权凭据。审批字典是教学占位符，不代表 IAM 身份认证。

**注意 TOCTOU**：检查通过与提交之间可能有变化，本课在切换指针前重新核对快照与容量，仍明确不宣称多进程原子性。

In [14]:
import tempfile
with tempfile.TemporaryDirectory() as dry_run_dir:
    tok_demo, model_demo = build_servable_models(8304)
    cluster_demo = Coordinator(Path(dry_run_dir)/'cluster',tok_demo,model_demo,
                               policy='sync',seed=8304,capacity_blocks=40)
    store_demo = ConfigStore(Path(dry_run_dir)/'config',cluster_demo)
    old_ptr,old_cfg = store_demo.read_active()
    new_cfg = changed_config(old_cfg,route_canary_percent=100)
    precheck = store_demo.preflight(new_cfg,tick=1)
    print('预检结果:', precheck['decision'])
    print('双副本探针最大误差:', precheck['probe']['max_cache_full_diff'])
    print('旧指针:', old_ptr)
    try:
        store_demo.publish(new_cfg,precheck,store_demo.approval(new_cfg,1),1,1,fail_after='stage_A')
    except SimulatedCrash as error:
        print('模拟单副本预置后故障:', error)
    print('故障后仍是旧指针:', store_demo.read_pointer()==old_ptr)
    saved = store_demo.publish(new_cfg,precheck,store_demo.approval(new_cfg,1),1,1)
    print('补全预置后成功提交:', saved['generation'])

预检结果: allow
双副本探针最大误差: 3.5762786865234375e-07
旧指针: {'generation': 1, 'sha': '795ea4246fcf4aa7aa724bdd865a686b862b80b8b78ee8e059d42501ea2e3f4b'}


模拟单副本预置后故障: 只预置了部分副本：活动指针未改变，不能宣称发布成功
故障后仍是旧指针: True
补全预置后成功提交: 2


## 8. 检测漂移和恢复配置：不是自动抹掉差异

本课将 A/B 两份副本配置与活动指针的期望版本比较：

```text
期望：telemetry_sample_percent = 20
观测：副本A = 20，副本B = 99
```

即使模型还能正常解码，**任何副本观测配置漂移都阻断新流量和自动接管**。先记录路径、严重度和来源，再用明确绑定版本的审批记录修复目标副本。不能把“重写期望文件让漂移消失”称为修复，也不能用修复配置来掩盖运行模型权重已改变。

NIST 将配置基线、变更控制、监控发现漂移作为持续管理过程，而不是一次性部署成功就结束。[NIST SP 800-128](https://csrc.nist.gov/pubs/sp/800/128/upd1/final)

## 9. 故障接管不只要选出一个主节点

恢复前必须核对：活动配置与目标副本、sync ACK 与 standby 日志前缀、模型权重/模板、容量与剩余KV预算、epoch和逻辑RTO。

1. **先预检，后模拟主机关闭**：小RTO预算必须在触碰健康实例前拒绝。
2. 从已验证的持久WAL重新构造逻辑会话，再用完整 token 前缀在目标上重建物理KV。
3. 新epoch切换后，旧epoch写入被围栏；旧主需要重新从可信日志恢复为备机。
4. 同一幂等键使用最早WAL中已固定的 stable/canary，不被新路由覆盖。
5. 终态检查两个CPU物理KV池都归零；不能只判断逻辑会话结束。

本课故障是**单进程模拟主机不可用**，不触碰真实主机/网络。RPO 与 RTO 都是教学日志序号与逻辑tick，不是对生产恢复时延的承诺。

In [15]:
# %% 11 故障接管的发布配置校验和端到端KV验收
class RecoveryGate:
    def __init__(self,store:ConfigStore):self.store=store

    def preflight(self,target:str,tick:int,ttl_ticks:int=4) -> dict:
        store=self.store;cluster=store.cluster
        require(target in {'A','B'} and target!=cluster.control['active'],'目标应是另一个副本')
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '逻辑时间/TTL非法')
        p,s=store.read_active(require_healthy=True)
        require(s.wal_policy=='sync','本课端到端零RPO验收要求sync')
        standby=cluster.rep[target]
        require(standby.up,'接管目标不可用')
        require(standby.ledger.seq>=cluster.control['ack_seq'],
                '目标没有全部已ACK日志，不能无损接管')
        require(standby.journal.records==cluster.rep[cluster.control['active']].journal.records[:len(standby.journal.records)],
                '接管目标日志不是当前主的前缀')
        capacity=store.safe_capacity(s)[target]
        audit=standby.audit();require(audit['max_cache_full_logits_diff']<=2e-5,
                                      '真实KV与全上下文输出不一致')
        payload={'kind':'recovery_preflight','generation':p['generation'],
                 'config_sha':p['sha'],'epoch':cluster.control['epoch'],
                 'old_active':cluster.control['active'],'target':target,
                 'ack_seq':cluster.control['ack_seq'],'target_seq':standby.ledger.seq,
                 'target_head':standby.ledger.last_hash,'control_sha':digest(cluster.control),
                 'issued_tick':tick,'expires_tick':tick+ttl_ticks,'capacity':capacity,
                 'target_kv_max_abs_diff':audit['max_cache_full_logits_diff'],
                 'target_active_sessions':audit['active_sessions'],
                 'estimated_rto_logical_ticks':2+math.ceil(len(standby.journal.records)/16)+
                    math.ceil(sum(len(st.prompt_ids)+len(st.generated) for st in standby.ledger.sessions.values()
                                  if st.terminal is None)/32),
                 'decision':'allow'}
        payload['seal']=digest(payload)
        return payload

    def promote(self,proof:dict,tick:int,rto_budget_ticks:int=40) -> dict:
        store=self.store;cluster=store.cluster
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '恢复门槛证据被修改')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '恢复证据已过期')
        require(proof['decision']=='allow','恢复证据未放行')
        p,s=store.read_active(require_healthy=True)
        require(p['generation']==proof['generation'] and p['sha']==proof['config_sha'],
                '恢复前配置代际被改动')
        require(cluster.control['epoch']==proof['epoch'] and
                cluster.control['active']==proof['old_active'] and
                cluster.control['ack_seq']==proof['ack_seq'] and
                digest(cluster.control)==proof['control_sha'],
                '恢复前控制epoch/ACK发生变化，必须重新取证')
        target=proof['target']
        require(cluster.rep[target].ledger.seq==proof['target_seq'] and
                cluster.rep[target].ledger.last_hash==proof['target_head'],
                '恢复目标日志被改动')
        store.safe_capacity(s)
        require(type(rto_budget_ticks) is int and rto_budget_ticks>=proof['estimated_rto_logical_ticks'],
                '恢复逻辑RTO预算过小；不允许先关闭健康主机再失败')
        # 明确模拟主机宕机，但并不接触真实主机或外部网络。
        cluster.rep[cluster.control['active']].shutdown()
        result=cluster.promote(proof['epoch'],target,rpo_budget_ticks=0,
                               rto_budget_ticks=rto_budget_ticks)
        require(result['lost_acked_events']==0 and result['rpo_gap_logical_ticks']==0,
                '切主结果不满足零RPO')
        require(cluster.control['epoch']==proof['epoch']+1 and
                cluster.control['active']==target,'旧主没有被epoch隔离')
        store.assert_aligned()
        audit=cluster.rep[target].audit()
        require(audit['max_cache_full_logits_diff']<=2e-5,'重新计算的KV与全前向不一致')
        check={'failover':result,'active':target,'epoch':cluster.control['epoch'],
               'old_epoch_fenced':True,'actual_kv_rebuilt':True,
               'max_kv_full_logits_diff':audit['max_cache_full_logits_diff'],
               'active_sessions_rebuilt':audit['active_sessions'],
               'config_generation_preserved':store.read_pointer()['generation']==proof['generation'],
               'zero_rpo_verified':True,'logical_time_only':True}
        store.audit.add(tick,'recovery_accepted',{'generation':proof['generation'],
                                                'epoch':cluster.control['epoch'],'ack_seq':cluster.control['ack_seq']})
        return check

## 10. 全流程独立演练：发布 → 漂移 → 预置失败 → 接管 → 清理

真实运行如下六类事件：

- 初始 stable 请求，发布为100% canary，重试仍用 stable；
- 已发证据遭遇新的 ACK，旧预检必须拒绝；
- 副本 B 的观测配置被修改，路由拒绝；凭显式修复记录恢复；
- 预置 A 后故障，活动指针不变；重试补全预置才提交；
- 双副本日志满足零RPO后，模拟主机丢失并冷恢复真实KV；
- 以新 epoch 继续同步服务，结束所有会话，双副本物理KV释放归零。

不会在展示报告时伪造模型能力或任何真实云平台测量。

In [16]:
# %% 12 六类变更/恢复场景：真实CPU KV、WAL和双副本配置

def finish_request(store:ConfigStore,rid:str) -> list[dict]:
    events=[]
    for _ in range(5):
        active=store.cluster.rep[store.cluster.control['active']]
        st=active.ledger.sessions[rid]
        if st.terminal is not None:break
        events.append(store.step(rid))
    require(store.cluster.rep[store.cluster.control['active']].ledger.sessions[rid].terminal is not None,
            '请求在预算内没有达到终态')
    return events


def run_experiment(output_dir:Path=Path('lesson_83_outputs'),seed:int=8301) -> Path:
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    root=Path(output_dir)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    root.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'status':'running','python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','seed':seed,'sources':SOURCES,
              'limitations':['single_process_serialized_controller_not_distributed_consensus',
                 'only_local_fsync_no_remote_durability','approval_is_plain_simulated_record_not_IAM',
                 'hash_chain_not_external_signature','logical_ticks_not_seconds',
                 'toy_random_LM_not_pretrained','no_external_faults','not_exact_GPU_benchmark']}
    write_json(root/'manifest.json',manifest)
    events={}
    try:
        tok,models=build_servable_models(seed)
        cluster=Coordinator(root/'cluster',tok,models,policy='sync',seed=seed,capacity_blocks=64)
        store=ConfigStore(root/'config',cluster)
        p0,s0=store.read_active();events['initial']={'pointer':p0,'config':s0.wire(),
                                                  'actual_kv_probe':store.probe_kv()}
        # 第一个请求WAL明确绑定 stable，路由后来切换不能改掉这一请求。
        first=store.route('alpha','pin01','hello',4)
        require(first['version']=='stable' and first['new_event'],'初始路由不符合stable')
        candidate=changed_config(s0,route_canary_percent=100,telemetry_sample_percent=20)
        proof=store.preflight(candidate,1)
        release=store.publish(candidate,proof,store.approval(candidate,1),1,1)
        retry=store.route('alpha','pin01','hello',4)
        second=store.route('beta','canary01','hello',4)
        require(not retry['new_event'] and retry['version']=='stable' and
                second['new_event'] and second['version']=='canary','版本固定或新路由错误')
        events['release']={'preflight_sha':proof['seal'],'commit':release,
                           'first_version':first['version'],'retry_version':retry['version'],
                           'retry_reused_wal':not retry['new_event'],
                           'next_request_version':second['version']}
        # 旧证据因ACK水位变更失效。轮换配置不得“复用曾经放行”的证据。
        s2=store.read_active()[1]
        next_cfg=changed_config(s2,telemetry_sample_percent=21)
        stale=store.preflight(next_cfg,2)
        third=store.route('alpha','drift_probe','hello',4)
        try:store.publish(next_cfg,stale,store.approval(next_cfg,2),2,2)
        except LessonError as exc:events['stale_evidence']={'blocked':True,'reason':str(exc),
                                              'new_ack_seq':cluster.control['ack_seq']}
        else:raise AssertionError('ACK变化后不能复用旧发布证据')
        # 恶意/意外的B副本配置漂移；不由自动reconciler悄悄掩盖。
        active_p,active_cfg=store.read_active()
        bad=changed_config(active_cfg,telemetry_sample_percent=99)
        atomic_json(store._replica_path('B',active_p['generation']),bad.wire())
        detected=store.detect_drift()
        require(not detected['healthy'] and any(x['replica']=='B' for x in detected['findings']),
                '检测不到B副本漂移')
        try:store.route('alpha','must_block','hello',4)
        except LessonError:pass
        else:raise AssertionError('漂移期间新请求必须拒绝')
        repaired=store.reconcile('B',store.approval(active_cfg,active_p['generation']),3,
                                 active_p['generation'])
        events['drift']={'detected':detected,'reconciled':repaired,
                         'after':store.detect_drift()}
        # 发布中途故障不能改变活动指针；再用相同证据重试预置，只提交一次。
        candidate3=changed_config(active_cfg,telemetry_sample_percent=25)
        prepare=store.preflight(candidate3,4)
        before=store.read_pointer()
        try:store.publish(candidate3,prepare,store.approval(candidate3,2),2,4,fail_after='stage_A')
        except SimulatedCrash:pass
        else:raise AssertionError('应注入A预置后故障')
        require(store.read_pointer()==before,'仅单副本预置时不得生效')
        result3=store.publish(candidate3,prepare,store.approval(candidate3,2),2,4)
        events['staged_failure']={'pointer_unchanged_until_commit':True,
                                  'committed_generation':result3['generation']}
        # 接管前必须根据已经实际发生的ACK及配置进行恢复门槛验证。
        recovery=RecoveryGate(store)
        recproof=recovery.preflight('B',5)
        # 先创建旧epoch的命令意图，再严格检查控制序号。
        old_epoch=cluster.control['epoch']
        recovered=recovery.promote(recproof,5,40)
        require(recovered['zero_rpo_verified'] and recovered['config_generation_preserved'],
                '端到端恢复验收失败')
        try:cluster._fence(old_epoch)
        except LessonError:pass
        else:raise AssertionError('旧epoch必须拒绝')
        # 由可信本地持久WAL冷重建旧主作为新备机，使同步ACK继续可用。
        cluster.rep['A']=Replica('A',cluster.root/'A',tok,models,cluster.capacity_blocks,load_existing=True)
        cluster.audit()
        same=store.route('alpha','pin01','hello',4)
        require(not same['new_event'] and same['version']=='stable',
                '故障切主后原请求版本必须从持久WAL恢复')
        after=store.route('alpha','after_dr','hello',4)
        require(after['version']=='canary' and after['new_event'],'故障恢复后新路由未恢复')
        events['recovery']={**recovered,'old_request_version_after_failover':same['version'],
                            'new_request_version_after_failover':after['version']}
        # 结束所有活动请求，验证真实物理KV块释放；不只清空Python计数。
        active=cluster.rep[cluster.control['active']]
        ids=[rid for rid,st in sorted(active.ledger.sessions.items()) if st.terminal is None]
        final_tokens={rid:len(finish_request(store,rid)) for rid in ids}
        last=cluster.audit()
        for name in ('A','B'):
            require(cluster.rep[name].pool.audit()['live_blocks']==0,
                    '终态仍有物理KV页，不能宣布恢复结束')
        events['final']={'active':last['active'],'epoch':last['epoch'],
                         'finished_requests':len(final_tokens),
                         'generated_steps':final_tokens,'physical_kv_zero':True,
                         'wal_durability':cluster.control['policy'],
                         'revision':store.read_pointer()['generation'],
                         'audit':store.audit.verify()}
        write_json(root/'experiment.json',events)
        reload_report=reload_run(root)
        write_json(root/'reload.json',reload_report)
        manifest['status']='completed';write_json(root/'manifest.json',manifest)
        print('发布版本',store.read_pointer()['generation'],'恢复epoch',last['epoch'],
              '实际KV页归零',events['final']['physical_kv_zero'])
        print('报告目录:',root)
        return root
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(root/'manifest.json',manifest)
        raise


def reload_run(folder:Path) -> dict:
    folder=Path(folder)
    tok,models=load_model_artifacts(folder/'cluster'/'models.pt')
    control=json.loads((folder/'cluster'/'control.json').read_text(encoding='utf-8'))
    cluster=Coordinator(folder/'cluster',tok,models,policy=control['policy'],
                        seed=control['seed'],capacity_blocks=control['capacity_blocks'],
                        load_existing=True)
    store=ConfigStore(folder/'config',cluster,load_existing=True)
    p,s=store.read_active()
    evidence=store.audit.verify()
    require(len(evidence['events'])>=3,'变更审计不完整')
    require(evidence['events'][0]=='initial_release' and
            'release_committed' in evidence['events'] and
            'recovery_accepted' in evidence['events'],
            '发布与恢复没有完整审计事件')
    check=cluster.audit()
    require(check['active_audit']['max_cache_full_logits_diff']<2e-5,
            '冷恢复真实KV与全上下文不一致')
    require(p['generation']>=2 and check['epoch']>=2,'并未完成所需发布/恢复')
    result={'generation':p['generation'],'config_sha':p['sha'],
            'model_sha':digest(cluster.model_identity),'epoch':check['epoch'],
            'cold_recovery_kv_max_abs_diff':check['active_audit']['max_cache_full_logits_diff'],
            'wal_ack_seq':check['ack_seq'],'change_events':evidence['count'],
            'audit_head':evidence['head'],'physical_kv_restored_from_disk':False,
            'logical_sessions_recovered':check['active_audit']['logical_requests'],
            'all_recovered_sessions_terminal':check['active_audit']['active_sessions']==0,
            'no_real_cluster_claim':True}
    return result

In [17]:
RUN_DIR = None
if RUN_FULL_LAB:
    RUN_DIR = run_experiment(OUTPUT_ROOT,seed=8301)
else:
    print('演练已关闭；不展示伪造的报告。')

发布版本 3 恢复epoch 2 实际KV页归零 True
报告目录: lesson_83_outputs/run_20261105_080936_785228Z


In [18]:
if RUN_DIR is not None:
    result = json.loads((RUN_DIR/'experiment.json').read_text(encoding='utf-8'))
    print('初始与发布:', result['release']['first_version'], '→',
          result['release']['next_request_version'])
    print('重试仍用:', result['release']['retry_version'])
    print('发布证据失效:', result['stale_evidence']['blocked'])
    print('漂移检测:', result['drift']['detected']['healthy'], '修复后:', result['drift']['after']['healthy'])
    print('部分预置故障不改变指针:', result['staged_failure']['pointer_unchanged_until_commit'])
    print('恢复结果:', {k:result['recovery'][k] for k in
           ['epoch','zero_rpo_verified','actual_kv_rebuilt','old_request_version_after_failover']})
    print('最终物理KV归零:', result['final']['physical_kv_zero'])

初始与发布: stable → canary
重试仍用: stable
发布证据失效: True
漂移检测: False 修复后: True
部分预置故障不改变指针: True
恢复结果: {'epoch': 2, 'zero_rpo_verified': True, 'actual_kv_rebuilt': True, 'old_request_version_after_failover': 'stable'}
最终物理KV归零: True


## 11. 审计链不等于外部可信签名

本课的 `changes.jsonl` 用事件序号、前一个哈希、当前摘要组成链条，另写本地 head。篡改中间字段或删掉尾部而不更新 head 会被发现。

但拥有两份文件写权限的攻击者可以同时重写链与 head。我们**不宣称 hash 本身能验证操作者身份或防止恶意完全重写**。生产环境还需要独立身份系统、外部锚点、访问控制与不可改写审计存储。

课程报告只存版本、摘要、状态计数和模拟事件名，不在变更审计中保存原始用户提示、token数组或API密钥。

In [19]:
if RUN_DIR is not None:
    current = json.loads((RUN_DIR/'reload.json').read_text(encoding='utf-8'))
    print('独立恢复代际/epoch:', current['generation'], current['epoch'])
    print('冷恢复真实KV与全前向最大误差:', current['cold_recovery_kv_max_abs_diff'])
    print('审计事件数:',current['change_events'])
    print('没有从磁盘反序列化物理KV张量:',not current['physical_kv_restored_from_disk'])

独立恢复代际/epoch: 3 2
冷恢复真实KV与全前向最大误差: 0.0
审计事件数: 5
没有从磁盘反序列化物理KV张量: True


## 12. 自动检查：真正的拒绝比“可以运行”更重要

固定检查覆盖：schema字段与类型、租户/模型隔离、容量、发布证据绑定、模拟审批、旧代际CAS、部分预置、提交后ACK丢失、漂移检测、真实权重篡改、WAL冻结、旧epoch围栏、恢复逻辑RTO与物理KV归零。

包含 **“提交了活动指针但没收到ACK”** 的独立测试：在审计补偿前拒绝流量，读取实际版本，对账后补写审计，但不再次递增代际。

本课不将自动检查通过等同于真实生产系统的IAM、远端持久性、分布式共识、实时SLO或高可用性证明。

In [20]:
# %% 13 固定检查：必须实际失败的门槛，和实际成功的恢复路径
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,a,b):self.ok(name,a==b)
    def close(self,name,a,b,tol=2e-5):self.ok(name,abs(a-b)<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,SimulatedCrash):self.names.append(name);return
        raise AssertionError(name+'：原本应阻断')


def run_checks() -> dict:
    torch.set_num_threads(1);check=Checks()
    tok,models=build_servable_models(8302)
    check.ok('模型与tokenizer均固定CPU',all(next(m.parameters()).device.type=='cpu' for m in models.values()))
    check.ok('稳定与灰度是不同的LoRA权重',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
    check.equal('模型版本集合',set(models),{'stable','canary'})
    check.equal('chat模板包含生成头',tok.apply_chat_template([{'role':'user','content':'hello'}],add_generation_prompt=True)[-2:],
                [tok.im_start_token_id,tok.role_ids['assistant']])
    check.ok('模型本身不可训练',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    with tempfile.TemporaryDirectory() as temp:
        root=Path(temp)
        c=Coordinator(root/'cluster',tok,models,policy='sync',seed=8302,capacity_blocks=40)
        store=ConfigStore(root/'config',c)
        p0,s0=store.read_active()
        check.equal('初始配置代际是1',p0['generation'],1)
        check.equal('初始路由无灰度',s0.route_canary_percent,0)
        check.equal('初始WAL同步',s0.wal_policy,'sync')
        check.ok('初始模型身份绑定实际权重',s0.model_identity_hash==digest(artifact_identity(tok,models)))
        check.equal('初始模板契约绑定真实tokenizer',s0.tokenizer_template_hash,digest(tok.contract()))
        check.ok('活动指针验证了双副本',len(store.assert_aligned()['replicas'])==2)
        check.equal('初始审计仅初始化',store.audit.verify()['count'],1)
        probes=store.probe_kv()
        check.ok('真实KV对全前向输出一致',probes['max_cache_full_diff']<=2e-5)
        check.ok('4套真实临时KV都释放',all(x['released'] for x in probes['checked'].values()))
        check.equal('真实探针包含两副本和两LoRA',len(probes['checked']),4)
        check.ok('安全容量预算有余量',store.safe_capacity(s0)['A']['headroom_blocks']>0)
        check.equal('配置JSON按类型往返',ServingConfig.from_wire(s0.wire()).wire(),s0.wire())
        check.equal('相同配置diff为空',config_diff(s0,s0),[])
        check.equal('单项路由变更diff路径', [d['path'] for d in config_diff(s0,changed_config(s0,route_canary_percent=25))],
                    ['route_canary_percent'])
        check.ok('租户变更归为严重漂移',config_diff(s0,changed_config(s0,allowed_tenants=['alpha']))[0]['severity']=='critical')
        check.ok('可观测性百分比属于受控变更',config_diff(s0,changed_config(s0,telemetry_sample_percent=30))[0]['severity']=='controlled')
        bad=s0.wire();bad['unexpected']=7
        check.rejects('拒绝未知配置字段',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad.pop('wal_policy')
        check.rejects('拒绝缺失配置字段',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['route_canary_percent']=True
        check.rejects('拒绝bool冒充路由百分比',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['reserve_blocks']=True
        check.rejects('拒绝bool冒充KV页容量',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['allowed_tenants']=['beta','alpha']
        check.rejects('拒绝未排序租户名单',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['allowed_tenants']=['alpha','alpha']
        check.rejects('拒绝重复租户',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['model_identity_hash']='fake'
        check.rejects('拒绝伪模型身份',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['wal_policy']='async'
        check.rejects('发布零RPO时拒绝静默async',lambda:ServingConfig.from_wire(bad))
        bad=s0.wire();bad['admission_enabled']=1
        check.rejects('拒绝整数冒充显式准入开关',lambda:ServingConfig.from_wire(bad))
        check.rejects('拒绝低于最低余量',lambda:changed_config(s0,reserve_blocks=1))
        check.rejects('拒绝百分比超100',lambda:changed_config(s0,route_canary_percent=101))
        check.rejects('拒绝不安全路由salt',lambda:changed_config(s0,routing_salt='../../etc'))
        check.rejects('拒绝模型身份字段修改',lambda:store.preflight(changed_config(s0,model_identity_hash='a'*64),0))
        check.rejects('拒绝模板hash字段修改',lambda:store.preflight(changed_config(s0,tokenizer_template_hash='a'*64),0))
        check.rejects('拒绝物理页大于预算',lambda:store.preflight(changed_config(s0,reserve_blocks=41),0))
        check.rejects('拒绝TTL 0',lambda:store.preflight(s0,0,ttl_ticks=0))
        check.rejects('拒绝负逻辑时间',lambda:store.preflight(s0,-1))
        check.rejects('拒绝未授权租户',lambda:store.route('gamma','g1'))
        check.rejects('拒绝不在允许范围的生成长度',lambda:store.route('alpha','k1',max_new_tokens=5))
        first=store.route('alpha','first','hello',4)
        check.equal('真实推理创建stable会话',first['version'],'stable')
        check.ok('创建事件已经双副本ACK',first['ack']['durability']=='primary+standby')
        check.equal('创建后的同步ACK水位',c.control['ack_seq'],1)
        check.equal('两副本日志序号相同',c.rep['A'].ledger.seq,c.rep['B'].ledger.seq)
        check.equal('真实KV包含请求会话',len(c.rep['A'].pool.sessions),1)
        check.ok('副本A增量KV与全重算一致',c.rep['A'].audit()['max_cache_full_logits_diff']<=2e-5)
        check.ok('副本B增量KV与全重算一致',c.rep['B'].audit()['max_cache_full_logits_diff']<=2e-5)
        again=store.route('alpha','first','hello',4)
        check.ok('同幂等键重试不会重复WAL',not again['new_event'])
        check.equal('同键重试使用原序号',again['ack']['seq'],first['ack']['seq'])
        check.rejects('同幂等键变更prompt拒绝',lambda:store.route('alpha','first','world',4))
        check.rejects('存活租户不能直接移出ACL',lambda:store.preflight(changed_config(s0,allowed_tenants=['beta']),1))
        candidate=changed_config(s0,route_canary_percent=100,telemetry_sample_percent=20)
        proof=store.preflight(candidate,1)
        check.equal('发布证据绑定代际',proof['base']['pointer']['generation'],1)
        check.equal('发布证据绑定ACK',proof['base']['ack_seq'],1)
        check.ok('发布证据有双副本真实KV探针',len(proof['probe']['checked'])==4)
        check.ok('发布证据绑定差异',set(x['path'] for x in proof['diff'])=={'route_canary_percent','telemetry_sample_percent'})
        check.equal('发布证据摘要正确',proof['seal'],digest({k:v for k,v in proof.items() if k!='seal'}))
        check.rejects('拒绝过期证据',lambda:store.verify_preflight(proof,candidate,6))
        check.rejects('拒绝未生效tick',lambda:store.verify_preflight(proof,candidate,0))
        changed=copy.deepcopy(proof);changed['issued_tick']=0
        check.rejects('篡改证据有效期拒绝',lambda:store.verify_preflight(changed,candidate,1))
        check.rejects('把其他候选绑定到旧证据拒绝',lambda:store.verify_preflight(proof,s0,1))
        check.rejects('错CAS版本拒绝',lambda:store.publish(candidate,proof,store.approval(candidate,1),0,1))
        check.rejects('未来代际不允许跳号发布',
                      lambda:store.publish(candidate,proof,store.approval(candidate,2),2,1))
        check.rejects('无模拟审批拒绝',lambda:store.publish(candidate,proof,{},1,1))
        check.rejects('审批不绑定候选拒绝',lambda:store.publish(candidate,proof,store.approval(s0,1),1,1))
        check.rejects('拒绝错误审批代际',lambda:store.publish(candidate,proof,store.approval(candidate,2),1,1))
        check.rejects('拒绝部分预置故障',lambda:store.publish(candidate,proof,store.approval(candidate,1),1,1,'stage_A'))
        check.equal('部分预置不修改活动指针',store.read_pointer(),p0)
        check.ok('部分预置文件不自动生效',store._version_path(2).exists() and not store._replica_path('B',2).exists())
        result=store.publish(candidate,proof,store.approval(candidate,1),1,1)
        check.equal('正式版本提交到2',result['generation'],2)
        check.ok('双副本变更后对齐',store.detect_drift()['healthy'])
        check.ok('正式提交产生可核对的审计',store._has_commit_receipt(2,result['active_sha']))
        check.rejects('不能重用第一次CAS覆盖第二次',lambda:store.publish(candidate,proof,store.approval(candidate,1),1,1))
        pinned=store.route('alpha','first','hello',4)
        check.equal('已存在请求仍使用stable',pinned['version'],'stable')
        check.ok('同键重试确实没有重复执行',not pinned['new_event'])
        new=store.route('beta','new','hello',4)
        check.equal('新请求路由canary',new['version'],'canary')
        check.equal('原stable会话KV仍绑定stable',c.rep['A'].pool.sessions[first['rid']].version,'stable')
        check.equal('新canary会话KV绑定canary',c.rep['A'].pool.sessions[new['rid']].version,'canary')
        check.equal('路由配置代际2',new['config_generation'],2)
        other_cfg=changed_config(candidate,telemetry_sample_percent=23)
        old_proof=store.preflight(other_cfg,2)
        _=store.step(first['rid'])
        check.rejects('ACK变化导致发布证据失效',lambda:store.publish(other_cfg,old_proof,store.approval(other_cfg,2),2,2))
        check.equal('证据失效并不修改活动配置',store.read_pointer()['generation'],2)
        # 改一个不影响执行的字段，依然阻断：配置漂移检测不能只覆盖权重。
        drifted=changed_config(candidate,telemetry_sample_percent=77)
        atomic_json(store._replica_path('B',2),drifted.wire())
        d=store.detect_drift()
        check.ok('观察到漂移',not d['healthy'])
        check.ok('识别B副本的字段路径',any(x.get('replica')=='B' and x.get('path')=='telemetry_sample_percent'
                                                  for x in d['findings']))
        check.rejects('B配置漂移拒绝新流量',lambda:store.route('alpha','blocked'))
        check.rejects('B配置漂移阻断恢复预检',lambda:RecoveryGate(store).preflight('B',3))
        check.rejects('B配置漂移阻断继续生成',lambda:store.step(new['rid']))
        check.rejects('无审批不能自动修复漂移',lambda:store.reconcile('B',{},3,2))
        repair=store.reconcile('B',store.approval(candidate,2),3,2)
        check.ok('显式reconcile后健康',repair['healthy'] and store.detect_drift()['healthy'])
        check.ok('修复写入审计',store.audit.verify()['events'][-1]=='drift_reconciled')
        # 真实权重漂移不能靠只修复配置字段掩盖。
        canary=c.rep['B'].models['canary']
        orig=adapter_snapshot(canary)
        with torch.no_grad():adapters(canary)[-1][1].add_(0.02)
        check.rejects('B实际LoRA权重漂移阻断发布',lambda:store.preflight(other_cfg,4))
        restore_adapter(canary,orig)
        check.ok('真实LoRA权重修复后模型身份恢复',artifact_identity(tok,c.rep['B'].models)==c.model_identity)
        # 有预检的候选仍会绑定快照，且未达上限时允许完成。
        proof3=store.preflight(other_cfg,4)
        commit3=store.publish(other_cfg,proof3,store.approval(other_cfg,2),2,4)
        check.equal('第三代配置提交',commit3['generation'],3)
        check.rejects('上一代证据不能用于新代际',lambda:store.verify_preflight(proof,other_cfg,4))
        check.ok('配置发布前后没有改变模型基座',base_hash(c.rep['A'].models['stable'])==base_hash(c.rep['B'].models['stable']))
        rec=RecoveryGate(store)
        recovery_proof=rec.preflight('B',5)
        check.equal('恢复证据绑定旧epoch',recovery_proof['epoch'],1)
        check.equal('恢复证据绑定版本3',recovery_proof['generation'],3)
        check.equal('恢复证据真实KV最大误差受限',recovery_proof['target_kv_max_abs_diff']<=2e-5,True)
        check.rejects('小RTO预算不会预先关闭健康主',lambda:rec.promote(recovery_proof,5,1))
        check.ok('RTO校验失败时旧主仍活着',c.rep['A'].up)
        check.rejects('恢复证据过期拒绝接管',lambda:rec.promote(recovery_proof,10,40))
        changed_proof=copy.deepcopy(recovery_proof);changed_proof['ack_seq']+=1
        check.rejects('修改恢复证据必须拒绝',lambda:rec.promote(changed_proof,5,40))
        recovered=rec.promote(recovery_proof,5,40)
        check.ok('恢复通过真实KV重算',recovered['actual_kv_rebuilt'])
        check.ok('同步零RPO',recovered['zero_rpo_verified'])
        check.equal('恢复后新epoch为2',c.control['epoch'],2)
        check.equal('恢复后新主为B',c.control['active'],'B')
        check.equal('切主不改发布配置代际',store.read_pointer()['generation'],3)
        check.rejects('旧epoch围栏拒绝新写入',lambda:c._fence(1))
        # 旧主冷重建成备机后，才能继续双副本持久ACK。
        c.rep['A']=Replica('A',c.root/'A',tok,models,c.capacity_blocks,load_existing=True)
        check.ok('冷重建原主KV与全前向一致',c.rep['A'].audit()['max_cache_full_logits_diff']<=2e-5)
        check.equal('已接管会话版本固定',store.route('alpha','first','hello',4)['version'],'stable')
        check.ok('故障恢复后新会话仍走canary',store.route('beta','after')['version']=='canary')
        ids=[rid for rid,st in c.rep['B'].ledger.sessions.items() if st.terminal is None]
        for rid in ids:finish_request(store,rid)
        check.equal('所有物理KV页彻底释放A',c.rep['A'].pool.audit()['live_blocks'],0)
        check.equal('所有物理KV页彻底释放B',c.rep['B'].pool.audit()['live_blocks'],0)
        check.equal('两个副本最终WAL序号一致',c.rep['A'].ledger.seq,c.rep['B'].ledger.seq)
        check.ok('所有会话进入终态',not any(st.terminal is None for st in c.rep['B'].ledger.sessions.values()))
        check.ok('变更审计至少包含一次发布及一次修复及一次接管',
                 {'release_committed','drift_reconciled','recovery_accepted'}<=set(store.audit.verify()['events']))
        audit_path=store.audit.path
        audit_before=audit_path.read_bytes()
        # 变更审计不包含原始prompt和token IDs。
        check.ok('变更审计不包含用户提示',b'hello' not in audit_before and b'prompt' not in audit_before)
        check.equal('审计verify不改变文件',audit_path.read_bytes(),audit_before)
        lines=audit_before.splitlines();record=json.loads(lines[-1]);record['data']['epoch']=999
        audit_path.write_bytes(b'\n'.join(lines[:-1]+[canonical(record).encode()])+b'\n')
        check.rejects('审计行被改写即使JSON合法也检测',lambda:store.audit.verify())
        audit_path.write_bytes(audit_before)
        check.equal('恢复合法审计后摘要相同',store.audit.verify()['head'],
                    json.loads(store.audit.head_path.read_text())['head'])
        head_before=store.audit.head_path.read_bytes()
        store.audit.head_path.write_text(canonical({'count':999,'head':ZERO_HASH}))
        check.rejects('尾部被删/头被伪造不通过封存',lambda:store.audit.verify())
        store.audit.head_path.write_bytes(head_before)
        check.ok('审计完全复原',store.audit.verify()['count']>=4)
        # 冷恢复全部资源，独立模型对象重新计算KV，不加载旧tensor。
        ct,ms=load_model_artifacts(c.root/'models.pt')
        cold=Coordinator(c.root,ct,ms,policy='sync',seed=8302,capacity_blocks=40,load_existing=True)
        observed=ConfigStore(store.root,cold,load_existing=True)
        check.equal('冷恢复保持已发布代际',observed.read_pointer()['generation'],3)
        check.ok('冷恢复模型身份一致',observed.read_active()[1].model_identity_hash==digest(cold.model_identity))
        check.equal('冷恢复无活跃KV',cold.rep['B'].pool.audit()['live_blocks'],0)
        check.equal('冷恢复已确认ACK水位',cold.control['ack_seq'],c.control['ack_seq'])
    # 客户端没收到ACK：读真实活动指针进行一次性审计补偿，不当成失败发布重做。
    with tempfile.TemporaryDirectory() as temp:
        rr=Path(temp);tok,models=build_servable_models(8303)
        cc=Coordinator(rr/'cluster',tok,models,policy='sync',seed=8303,capacity_blocks=32)
        ss=ConfigStore(rr/'config',cc)
        p,cfg=ss.read_active()
        next_cfg=changed_config(cfg,route_canary_percent=100)
        proof=ss.preflight(next_cfg,1)
        approval=ss.approval(next_cfg,1)
        check.rejects('提交后ACK丢失明确抛出不确定结果',
                      lambda:ss.publish(next_cfg,proof,approval,1,1,fail_after='after_commit'))
        check.equal('ACK丢失但持久指针已经为2',ss.read_pointer()['generation'],2)
        check.rejects('缺少提交审计时拒绝服务流量',lambda:ss.route('alpha','x'))
        fixed=ss.resolve_uncertain_commit(next_cfg,proof,approval,1,1)
        check.ok('不确定ACK对账成功但不再次递增指针',fixed['committed'] and not fixed['pointer_advanced_again'])
        check.equal('对账后指针仍为2',ss.read_pointer()['generation'],2)
        check.ok('对账补充可查审计',ss._has_commit_receipt(2,ss.read_pointer()['sha']))
        check.ok('重复对账幂等',not ss.resolve_uncertain_commit(next_cfg,proof,approval,1,1)['audit_reconciled'])
        check.ok('对账后新请求才被允许',ss.route('alpha','x')['version']=='canary')
        check.rejects('ACK变更后旧对账凭证失效',lambda:ss.resolve_uncertain_commit(next_cfg,proof,approval,1,1))
    # 注入错误的恢复后健康报告，专门确认最终门槛不会只相信成功返回码。
    with tempfile.TemporaryDirectory() as temp:
        rr=Path(temp);tok,models=build_servable_models(8305)
        cc=Coordinator(rr/'cluster',tok,models,policy='sync',seed=8305,capacity_blocks=32)
        ss=ConfigStore(rr/'config',cc)
        ss.route('alpha','wip','hello',4)
        rg=RecoveryGate(ss)
        proof=rg.preflight('B',1)
        original_audit=Replica.audit
        def injected_bad_kv_report(replica):
            result=original_audit(replica)
            if replica.name=='B' and cc.control['active']=='B':
                result['max_cache_full_logits_diff']=0.5
            return result
        Replica.audit=injected_bad_kv_report
        try:
            check.rejects('切主后必须再次检查恢复KV健康阈值',lambda:rg.promote(proof,1,40))
        finally:
            Replica.audit=original_audit
    return {'passed':len(check.names),'names':check.names,
            'scope':'real_CPU_KV_two_local_replicas_WAL; simulated_single_controller_CAS; no_cloud_or_real_IAM'}

In [21]:
check_report = run_checks()
print('固定检查通过:',check_report['passed'],'项')
if RUN_DIR is not None:
    write_json(RUN_DIR/'checks.json',check_report)

固定检查通过: 134 项


## 13. 独立练习A：按字段识别漂移

实现 `drift_paths_ex(desired, observed)`：

- 必须拒绝两边键集合不一致；
- 值不同的字段名按字典序输出列表；
- 不能把字典插入顺序或整个JSON格式化字符串作为漂移依据；
- 不允许偷偷忽略新增的权限字段。

In [22]:
def check_exercise_a(fn):
    desired={'route_percent':10,'wal':'sync','telemetry':20,'tenants':['alpha','beta']}
    observed={'route_percent':20,'wal':'sync','telemetry':30,'tenants':['alpha','beta']}
    require(fn(desired,observed)==['route_percent','telemetry'],'A应报告排序的变化字段')
    require(fn(desired,copy.deepcopy(desired))==[],'A相同配置没有漂移')
    for bad in (lambda:fn(desired,{'wal':'sync'}),lambda:fn(desired,{'bad':3,**observed})):
        try:bad()
        except (LessonError,ValueError):pass
        else:raise LessonError('A不同字段集合必须拒绝')
    return {'exercise':'A','passed':True}


def check_exercise_b(fn):
    first=fn('alpha','idempotency001','routing83',10)
    require(first in {'stable','canary'} and first==fn('alpha','idempotency001','routing83',10),
            'B同输入必须得到相同版本')
    require(fn('alpha','idempotency001','routing83',0)=='stable','B 0%必须stable')
    require(fn('alpha','idempotency001','routing83',100)=='canary','B 100%必须canary')
    # 不依赖具体SHA值猜测：范围内大量键的0..100路由应形成不同且确定的结果。
    versions={fn('alpha','id'+str(i),'routing83',50) for i in range(32)}
    require(versions=={'stable','canary'},'B 50%应该覆盖两种路由')
    for bad in (lambda:fn('alpha','x','a',True),lambda:fn('alpha','x','a',101)):
        try:bad()
        except (LessonError,ValueError):pass
        else:raise LessonError('B非法百分比必须拒绝')
    return {'exercise':'B','passed':True}


def check_exercise_c(fn):
    context={'revision':2,'epoch':4,'ack_seq':11,'config_sha':'a'*64}
    proof={'context':copy.deepcopy(context),'issued_tick':10,'expires_tick':14}
    proof['seal']=digest(proof)
    require(fn(proof,context,12),'C同上下文、有效期内应通过')
    for changed_ctx,now in [(dict(context,ack_seq=12),12),(dict(context,epoch=5),12),
                            (context,15),(context,9)]:
        try:fn(proof,changed_ctx,now)
        except (LessonError,ValueError):pass
        else:raise LessonError('C旧水位/epoch或过期必须拒绝')
    bad=copy.deepcopy(proof);bad['context']['ack_seq']+=1
    try:fn(bad,context,12)
    except (LessonError,ValueError):pass
    else:raise LessonError('C证据篡改必须拒绝')
    return {'exercise':'C','passed':True}

In [23]:
def drift_paths_ex(desired, observed):
    raise NotImplementedError('请实现A：结构化字段漂移')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(check_exercise_a(drift_paths_ex))

## 14. 独立练习B：确定性哈希路由

实现 `route_bucket_ex(tenant,key,salt,percent)`：

- 使用 `digest({'tenant':tenant,'key':key,'salt':salt})` 的前12位十六进制数，对100取余；
- 当桶号小于percent时返回 `canary`，否则返回 `stable`；
- 0%始终 stable，100%始终 canary；
- 拒绝 bool 冒充百分比和超过100的百分比；
- 这只是新请求路由函数，**同键重试必须先检查WAL**。

In [24]:
def route_bucket_ex(tenant,key,salt,percent):
    raise NotImplementedError('请实现B：确定性路由')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(check_exercise_b(route_bucket_ex))

## 15. 独立练习C：预检证据与当前上下文绑定

实现 `validate_context_ex(proof,current,now)`：

- 先检查证据封装、字段集合与 SHA256；
- 预检里的 `revision, epoch, ack_seq, config_sha` 必须和当前上下文完全一致；
- `issued_tick ≤ now ≤ expires_tick`，边界允许；
- 任何字段篡改、旧ACK、旧epoch或过期均报错；
- 这只是**教学完整性检查**，没有真实IAM、电子签名或分布式快照。

In [25]:
def validate_context_ex(proof,current,now):
    raise NotImplementedError('请实现C：证据绑定和TTL')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(check_exercise_c(validate_context_ex))

## 16. 练习参考实现（请先独立作答）

参考实现采用独立函数名，不覆盖你自己的函数。完整发布流程还有身份与真实KV校验，三题仅训练各自的局部基础。

In [26]:
def answer_a(desired:dict,observed:dict):
    require(type(desired) is dict and type(observed) is dict and set(desired)==set(observed),
            '配置两侧必须拥有相同字段')
    return sorted(k for k in desired if desired[k]!=observed[k])


def answer_b(tenant:str,key:str,salt:str,percent:int):
    require(all(type(x) is str and bool(x) for x in (tenant,key,salt)) and
            type(percent) is int and 0<=percent<=100,'路由输入非法')
    bucket=int(digest({'tenant':tenant,'key':key,'salt':salt})[:12],16)%100
    return 'canary' if bucket<percent else 'stable'


def answer_c(proof:dict,current:dict,now:int):
    require(type(proof) is dict and type(current) is dict and type(now) is int,
            '证据/上下文类型非法')
    require(set(proof)=={'context','issued_tick','expires_tick','seal'} and
            proof['seal']==digest({k:v for k,v in proof.items() if k!='seal'}),
            '证据完整性不匹配')
    require(proof['context']==current,'证据上下文已变')
    require(type(proof['issued_tick']) is int and type(proof['expires_tick']) is int and
            proof['issued_tick']<=now<=proof['expires_tick'],'证据时间无效')
    return True

In [27]:
print('A:',check_exercise_a(answer_a))
print('B:',check_exercise_b(answer_b))
print('C:',check_exercise_c(answer_c))

A: {'exercise': 'A', 'passed': True}
B: {'exercise': 'B', 'passed': True}
C: {'exercise': 'C', 'passed': True}


## 17. 独立脚本命令、复现范围及下节预告

```bash
python lesson_83_serving_config_drift_release_gates.py --checks-only
python lesson_83_serving_config_drift_release_gates.py --math-only
python lesson_83_serving_config_drift_release_gates.py
python lesson_83_serving_config_drift_release_gates.py --reload-run lesson_83_outputs/实际运行目录
```

完整脚本仅依赖 PyTorch 和 Python 标准库，默认 CPU，不访问云端，不自动安装，不上传数据。重载只验证可信本地生成的工件与日志；不是恢复外部生产服务，也不是精确续训。

**本课验收**：能区分期望/生效配置与模型实际权重；能解释为什么ACK改变使发布预检失效；能证明“部分预置不生效、已提交未ACK需要对账”；能从WAL恢复原请求固定版本并验收真实KV；能说明单控制点串行CAS为何不能取代真实多节点共识。

**下一课建议：第84课——声明式控制器的漂移检测循环、幂等调谐与安全降级。** 把本课的手工发布门槛连接到定时调谐，但坚持不静默覆盖危险漂移。

## 参考资料（文档观点与本课实验数据分开）

- [Kubernetes Server-Side Apply：字段所有权和冲突](https://kubernetes.io/docs/reference/using-api/server-side-apply/)
- [NIST SP 800-128：安全聚焦的配置基线、变更控制与监控](https://csrc.nist.gov/pubs/sp/800/128/upd1/final)
- [Google SRE：Canarying Releases](https://sre.google/workbook/canarying-releases/)
- [Google SRE：可靠产品发布与逐步验证](https://sre.google/sre-book/reliable-product-launches/)
- [PyTorch：随机数与复现局限](https://docs.pytorch.org/docs/stable/notes/randomness.html)

本课演练是自行构造的CPU微型模型、两个本地目录和逻辑时钟，文档中的生产控制能力不归因于本课实现。

In [28]:
# %% 15 CLI：默认真实演练、自检、公式示例及独立冷恢复

def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    group=parser.add_mutually_exclusive_group()
    group.add_argument('--checks-only',action='store_true')
    group.add_argument('--math-only',action='store_true')
    group.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_83_outputs'))
    parser.add_argument('--seed',type=int,default=8301)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2))
        return 0
    if args.math_only:
        print(json.dumps({'config_change_paths':answer_a({'route':0,'wal':'sync'},
                                                        {'route':100,'wal':'sync'}),
                          'route_0':answer_b('alpha','k','course83',0),
                          'route_100':answer_b('alpha','k','course83',100),
                          'ttl_logic':'issued_tick <= now <= expires_tick',
                          'no_walltime_or_cloud_claim':True},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['passed']}项。单进程本地双副本/CPU真实KV，不是生产集群或IAM认证。")
    if args.checks_only:return 0
    folder=run_experiment(args.output_dir,args.seed)
    write_json(folder/'checks.json',checks)
    print('独立冷恢复:',json.dumps(reload_run(folder),ensure_ascii=False))
    return 0